# 🧬 Molecular Glues Diffusion Model – Full Pipeline Notebook

This self-contained notebook implements the **complete pipeline** for training and using a
graph-based discrete diffusion model to generate novel molecular glue candidates.

**Pipeline stages:**
1. **Setup** – install dependencies, mount Google Drive, configure device
2. **Configuration** – all hyperparameters in one place
3. **Utilities** – chemistry helpers, molecular filters, glue scoring
4. **Data** – graph representations, dataset loading, preprocessing
5. **Model** – noise scheduler, graph transformer, diffusion model
6. **Training** – trainer with EMA, LR scheduling, checkpointing
7. **Generation** – sampler, post-processing, diverse sampling
8. **Evaluation** – metrics, visualizations, full report
9. **Run** – end-to-end training → generation → analysis

> **Platforms:** Google Colab (with Drive mount) · DGX · any CUDA machine

---

## 1 · Environment Setup

In [ ]:
# ── Install dependencies ──────────────────────────────────────────
# Run this cell once per runtime to install all required packages.
# On Colab the base torch is pre-installed; we add the extras.

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.1.0+cu121.html
!pip install -q rdkit
!pip install -q scipy pandas seaborn tqdm matplotlib numpy

print("✅ All dependencies installed.")

### 1.1 · Google Drive Mount (optional)
Mount your Google Drive to persist checkpoints and data across sessions.

In [ ]:
# ── Google Drive mount ────────────────────────────────────────────
# Uncomment the block below if running on Google Colab and you want to
# save / load checkpoints and data from your Drive.

# from google.colab import drive
# drive.mount('/content/drive')

# Set these paths to your Drive folders:
DRIVE_DATA_DIR   = "/content/drive/MyDrive/molecular_glues/data"
DRIVE_CKPT_DIR   = "/content/drive/MyDrive/molecular_glues/checkpoints"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/molecular_glues/outputs"

# Local fallbacks (used when Drive is not mounted)
import os
DATA_DIR   = DRIVE_DATA_DIR   if os.path.isdir(DRIVE_DATA_DIR)   else "./data"
CKPT_DIR   = DRIVE_CKPT_DIR   if os.path.isdir(DRIVE_CKPT_DIR)   else "./checkpoints"
OUTPUT_DIR = DRIVE_OUTPUT_DIR  if os.path.isdir(DRIVE_OUTPUT_DIR)  else "./outputs"

for d in [DATA_DIR, CKPT_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Data dir:       {DATA_DIR}")
print(f"Checkpoint dir: {CKPT_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")

### 1.2 · Device Configuration

In [ ]:
# ── Device & seed ─────────────────────────────────────────────────
import torch
import numpy as np
import random

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2 · Configuration
All hyperparameters, vocabulary mappings, and dataclass-based config objects.

In [ ]:
"""
Configuration for Molecular Glue Diffusion Model
"""
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

@dataclass
class ModelConfig:
    """Graph transformer architecture settings."""
    hidden_dim: int = 256
    num_layers: int = 8
    num_heads: int = 8
    dropout: float = 0.1
    activation: str = "gelu"  # "gelu" or "silu"

    # Node features: atom types
    num_atom_types: int = 10  # C, N, O, S, F, Cl, Br, P, I, other
    num_charges: int = 5      # -2, -1, 0, +1, +2
    num_hybridizations: int = 4  # sp, sp2, sp3, other

    # Edge features: bond types
    num_bond_types: int = 5   # none, single, double, triple, aromatic

    # Property conditioning
    num_properties: int = 8   # MW, logP, aromatic_rings, HBD, HBA, Fsp3, TPSA, SA

@dataclass
class DiffusionConfig:
    """Diffusion process settings."""
    num_timesteps: int = 500
    beta_schedule: str = "cosine"  # "linear" or "cosine"
    beta_start: float = 1e-4
    beta_end: float = 0.02

@dataclass
class LossConfig:
    """Auxiliary loss weights."""
    lambda_valency: float = 3.0    # Increased from 0.5 to enforce chemical validity
    lambda_property: float = 1.0   # Increased from 0.3 to control HBD/HBA/TPSA
    lambda_fragment: float = 0.2   # Increased from 0.1

@dataclass
class GuidanceConfig:
    """Classifier-free guidance settings."""
    guidance_scale: float = 2.0
    condition_dropout: float = 0.1  # Probability of dropping condition during training

    # Target property ranges for glue-like generation
    target_mw: Tuple[float, float] = (300.0, 450.0)
    target_logp: Tuple[float, float] = (1.5, 3.5)
    target_aromatic_rings: Tuple[int, int] = (2, 3)
    target_hbd_hba: Tuple[int, int] = (4, 8)

@dataclass
class TrainingConfig:
    """Training hyperparameters."""
    batch_size: int = 64
    learning_rate: float = 1e-4
    weight_decay: float = 1e-6
    epochs: int = 100
    warmup_steps: int = 1000
    gradient_clip: float = 1.0

    # EMA
    ema_decay: float = 0.999

    # Validation
    val_split: float = 0.1
    val_frequency: int = 5  # validate every N epochs

    # Checkpointing
    checkpoint_dir: str = "checkpoints"
    save_frequency: int = 10

@dataclass
class GenerationConfig:
    """Molecule generation settings."""
    num_molecules: int = 1000
    max_atoms: int = 50
    min_atoms: int = 5
    batch_size: int = 100

    # Sampling
    temperature: float = 1.0

@dataclass
class MolecularConstraints:
    """Drug-like and glue-like property constraints."""
    # Molecular weight
    mw_min: float = 200.0
    mw_max: float = 500.0

    # Lipophilicity (tightened per spec)
    logp_min: float = 0.0
    logp_max: float = 4.0

    # Lipinski's Rule of Five
    hbd_max: int = 5   # H-bond donors
    hba_max: int = 10  # H-bond acceptors

    # Additional filters
    rotatable_bonds_max: int = 8
    tpsa_min: float = 20.0
    tpsa_max: float = 140.0

    # Synthetic accessibility (tightened per spec)
    sa_score_max: float = 4.5

    # Ring constraints
    min_rings: int = 1
    max_rings: int = 5
    min_aromatic_rings: int = 1

    # Rejection sampling
    max_heavy_atoms: int = 35
    min_fsp3: float = 0.2

@dataclass
class Config:
    """Master configuration."""
    model: ModelConfig = field(default_factory=ModelConfig)
    diffusion: DiffusionConfig = field(default_factory=DiffusionConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    generation: GenerationConfig = field(default_factory=GenerationConfig)
    constraints: MolecularConstraints = field(default_factory=MolecularConstraints)
    loss: LossConfig = field(default_factory=LossConfig)
    guidance: GuidanceConfig = field(default_factory=GuidanceConfig)

    # Paths
    data_dir: str = "data"
    output_dir: str = "output"

    # Device
    device: str = "cuda"  # "cuda" or "cpu"
    seed: int = 42

# Atom type vocabulary
ATOM_TYPES = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'P', 'I', 'Other']
ATOM_TO_IDX = {atom: idx for idx, atom in enumerate(ATOM_TYPES)}

# Bond type vocabulary
BOND_TYPES = ['NONE', 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']
BOND_TO_IDX = {bond: idx for idx, bond in enumerate(BOND_TYPES)}

# Charge vocabulary
CHARGES = [-2, -1, 0, 1, 2]
CHARGE_TO_IDX = {charge: idx for idx, charge in enumerate(CHARGES)}

# Hybridization vocabulary
HYBRIDIZATIONS = ['SP', 'SP2', 'SP3', 'OTHER']
HYBRID_TO_IDX = {h: idx for idx, h in enumerate(HYBRIDIZATIONS)}

# Property names for conditioning
PROPERTY_NAMES = [
    'molecular_weight', 'logp', 'num_aromatic_rings',
    'hbd', 'hba', 'fraction_sp3', 'tpsa', 'sa_score'
]

## 3 · Utility Modules
### 3.1 · Chemistry Utilities
Core RDKit helpers for validation, property calculation, fingerprints, and scaffolds.

In [ ]:
"""
Chemistry utility functions for molecular property calculation and validation.
"""
from typing import Optional, Dict, List
from rdkit import Chem
from rdkit.Chem import (
    Descriptors, rdMolDescriptors, AllChem,
    DataStructs, QED as QEDModule,
)
from rdkit.Chem.Scaffolds import MurckoScaffold
import numpy as np

def is_valid_molecule(mol_or_smiles) -> bool:
    """
    Check if a molecule or SMILES string is chemically valid.

    Args:
        mol_or_smiles: RDKit Mol object or SMILES string

    Returns:
        True if valid molecule
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        elif hasattr(mol_or_smiles, 'GetNumAtoms'):
            mol = mol_or_smiles
        else:
            return False

        if mol is None:
            return False

        # Try sanitization
        Chem.SanitizeMol(mol)
        return True
    except Exception:
        return False

def check_valency(mol_or_smiles) -> bool:
    """
    Check if all atoms have valid valency.

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        True if all valencies are correct
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        # Sanitize checks valency
        Chem.SanitizeMol(mol)
        return True
    except Chem.MolSanitizeException:
        return False
    except Exception:
        return False

def check_ring_stability(mol_or_smiles) -> bool:
    """
    Check if ring systems are stable (no 3-membered rings with double bonds, etc.).

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        True if ring systems are stable
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        ring_info = mol.GetRingInfo()
        for ring in ring_info.AtomRings():
            if len(ring) < 3:
                return False
            # 3-membered rings with double bonds are unstable
            if len(ring) == 3:
                for idx in ring:
                    atom = mol.GetAtomWithIdx(idx)
                    for bond in atom.GetBonds():
                        if bond.GetBondType() == Chem.BondType.DOUBLE:
                            other = bond.GetOtherAtomIdx(idx)
                            if other in ring:
                                return False
        return True
    except Exception:
        return False

def get_molecular_properties(mol_or_smiles) -> Optional[Dict[str, float]]:
    """
    Calculate molecular properties.

    Args:
        mol_or_smiles: RDKit Mol or SMILES string

    Returns:
        Dictionary of properties or None if invalid
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return None

        # Count ring Types
        ring_info = mol.GetRingInfo()
        num_rings = ring_info.NumRings()
        num_aromatic_rings = Descriptors.NumAromaticRings(mol)

        # Calculate Fsp3
        num_sp3 = 0
        num_carbons = 0
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 6:
                num_carbons += 1
                if atom.GetHybridization() == Chem.rdchem.HybridizationType.SP3:
                    num_sp3 += 1
        fraction_sp3 = num_sp3 / max(1, num_carbons)

        # QED
        qed_score = calculate_qed(mol)

        return {
            'molecular_weight': Descriptors.ExactMolWt(mol),
            'logp': Descriptors.MolLogP(mol),
            'hbd': rdMolDescriptors.CalcNumHBD(mol),
            'hba': rdMolDescriptors.CalcNumHBA(mol),
            'tpsa': Descriptors.TPSA(mol),
            'rotatable_bonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
            'num_rings': num_rings,
            'num_aromatic_rings': num_aromatic_rings,
            'num_heavy_atoms': mol.GetNumHeavyAtoms(),
            'fraction_sp3': fraction_sp3,
            'qed': qed_score,
        }
    except Exception:
        return None

def calculate_qed(mol_or_smiles) -> float:
    """
    Calculate Quantitative Estimate of Drug-likeness (QED).

    Args:
        mol_or_smiles: RDKit Mol or SMILES string

    Returns:
        QED score (0.0-1.0), 0.0 if invalid
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return 0.0

        return QEDModule.qed(mol)
    except Exception:
        return 0.0

def get_murcko_scaffold(smiles: str) -> Optional[str]:
    """
    Get the Murcko scaffold (generic framework) of a molecule.

    Args:
        smiles: SMILES string

    Returns:
        Canonical SMILES of the generic Murcko scaffold, or None
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        generic = MurckoScaffold.MakeScaffoldGeneric(scaffold)
        return Chem.MolToSmiles(generic, canonical=True)
    except Exception:
        return None

def canonicalize_smiles(smiles: str) -> Optional[str]:
    """
    Canonicalize a SMILES string.

    Args:
        smiles: Input SMILES

    Returns:
        Canonical SMILES or None if invalid
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def get_morgan_fingerprint(mol_or_smiles, radius: int = 2, n_bits: int = 2048):
    """
    Get Morgan (circular) fingerprint.

    Args:
        mol_or_smiles: RDKit Mol or SMILES
        radius: Fingerprint radius
        n_bits: Number of bits

    Returns:
        RDKit fingerprint object or None
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return None

        return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    except Exception:
        return None

def calculate_tanimoto_similarity(smiles1: str, smiles2: str) -> float:
    """
    Calculate Tanimoto similarity between two molecules.

    Args:
        smiles1: First molecule SMILES
        smiles2: Second molecule SMILES

    Returns:
        Tanimoto similarity (0.0-1.0)
    """
    try:
        fp1 = get_morgan_fingerprint(smiles1)
        fp2 = get_morgan_fingerprint(smiles2)

        if fp1 is None or fp2 is None:
            return 0.0

        return DataStructs.TanimotoSimilarity(fp1, fp2)
    except Exception:
        return 0.0

def calculate_tanimoto_batch(smiles: str, reference_smiles: List[str]) -> List[float]:
    """
    Calculate Tanimoto similarity of a molecule against a list of references.

    Args:
        smiles: Query molecule SMILES
        reference_smiles: List of reference SMILES

    Returns:
        List of Tanimoto similarities
    """
    fp = get_morgan_fingerprint(smiles)
    if fp is None:
        return [0.0] * len(reference_smiles)

    sims = []
    for ref in reference_smiles:
        ref_fp = get_morgan_fingerprint(ref)
        if ref_fp is not None:
            sims.append(DataStructs.TanimotoSimilarity(fp, ref_fp))
        else:
            sims.append(0.0)

    return sims

### 3.2 · Molecular Filters
Drug-likeness, glue-likeness, PAINS, SA score, and reactive-group filters.

In [ ]:
"""
Molecular filters for drug-likeness, glue-likeness, PAINS, and synthetic accessibility.
"""
from typing import Optional
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams
import math

def is_drug_like(mol_or_smiles) -> bool:
    """
    Check if molecule passes Lipinski's Rule of Five.

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        True if drug-like
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)

        violations = 0
        if mw > 500:
            violations += 1
        if logp > 5:
            violations += 1
        if hbd > 5:
            violations += 1
        if hba > 10:
            violations += 1

        return violations <= 1
    except Exception:
        return False

def is_glue_like(mol_or_smiles) -> bool:
    """
    Check if molecule has molecular glue-like properties.

    Criteria: MW 200-500, LogP 0-4, ≥1 aromatic ring, ≤8 rotatable bonds,
    TPSA 20-140, heavy atoms ≤35.

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        True if glue-like
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        num_aromatic_rings = Descriptors.NumAromaticRings(mol)
        rotatable_bonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
        tpsa = Descriptors.TPSA(mol)
        heavy_atoms = mol.GetNumHeavyAtoms()

        if mw < 200 or mw > 500:
            return False
        if logp < 0.0 or logp > 4.0:
            return False
        if num_aromatic_rings < 1:
            return False
        if rotatable_bonds > 8:
            return False
        if tpsa < 20.0 or tpsa > 140.0:
            return False
        if heavy_atoms > 35:
            return False

        return True
    except Exception:
        return False

def is_glue_target_range(mol_or_smiles) -> bool:
    """
    Check if molecule falls within the guided target ranges for glue-like generation.

    Tighter criteria: MW 300-450, LogP 1.5-3.5, 2-3 aromatic rings,
    HBD+HBA 4-8, Fsp3 > 0.2.
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        mw = Descriptors.ExactMolWt(mol)
        logp = Descriptors.MolLogP(mol)
        num_aromatic_rings = Descriptors.NumAromaticRings(mol)
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)

        # Fsp3
        num_sp3 = sum(1 for a in mol.GetAtoms()
                      if a.GetAtomicNum() == 6 and
                      a.GetHybridization() == Chem.rdchem.HybridizationType.SP3)
        num_carbons = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum() == 6)
        fsp3 = num_sp3 / max(1, num_carbons)

        if mw < 300 or mw > 450:
            return False
        if logp < 1.5 or logp > 3.5:
            return False
        if num_aromatic_rings < 2 or num_aromatic_rings > 3:
            return False
        if (hbd + hba) < 4 or (hbd + hba) > 8:
            return False
        if fsp3 < 0.2:
            return False

        return True
    except Exception:
        return False

def has_reactive_groups(mol_or_smiles) -> bool:
    """
    Check if molecule contains reactive functional groups that should be excluded.

    Checks for: aldehydes, epoxides, Michael acceptors, acyl halides,
    anhydrides, isocyanates.

    Returns:
        True if reactive groups are found (molecule should be rejected)
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return True

        reactive_smarts = [
            '[CH]=O',                       # Aldehyde
            'C1OC1',                        # Epoxide
            '[C]=[C][C]=O',                 # Michael acceptor (enone)
            'C(=O)[F,Cl,Br,I]',            # Acyl halide
            'C(=O)OC(=O)',                  # Anhydride
            'N=C=O',                        # Isocyanate
            'N=C=S',                        # Isothiocyanate
            'S(=O)(=O)F',                   # Sulfonyl fluoride
        ]

        for smarts in reactive_smarts:
            pattern = Chem.MolFromSmarts(smarts)
            if pattern and mol.HasSubstructMatch(pattern):
                return True

        return False
    except Exception:
        return True

def passes_pains_filter(mol_or_smiles) -> bool:
    """
    Check if molecule passes PAINS (Pan Assay INterference compoundS) filter.

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        True if passes (no PAINS alerts)
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return False

        params = FilterCatalogParams()
        params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
        catalog = FilterCatalog.FilterCatalog(params)

        return not catalog.HasMatch(mol)
    except Exception:
        return False

def calculate_sa_score(mol_or_smiles) -> Optional[float]:
    """
    Calculate Synthetic Accessibility (SA) score.

    Uses a simplified approach based on fragment contributions.
    Score ranges from 1 (easy) to 10 (hard).

    Args:
        mol_or_smiles: RDKit Mol or SMILES

    Returns:
        SA score (1-10) or None if invalid
    """
    try:
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None:
            return None

        # Complexity metrics
        num_rings = mol.GetRingInfo().NumRings()
        num_stereo = rdMolDescriptors.CalcNumAtomStereoCenters(mol)
        num_heavy = mol.GetNumHeavyAtoms()
        num_heteroatoms = sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in [6, 1])

        # Heuristic SA score
        score = 1.0
        score += 0.2 * num_rings
        score += 0.5 * num_stereo
        score += 0.02 * num_heavy
        score += 0.1 * num_heteroatoms
        score += 0.3 * len([b for b in mol.GetBonds() if b.GetBondType() == Chem.BondType.TRIPLE])

        # Bridged rings are harder
        ring_info = mol.GetRingInfo()
        if ring_info.NumRings() > 1:
            atom_rings = ring_info.AtomRings()
            for i, ring_a in enumerate(atom_rings):
                for ring_b in atom_rings[i + 1:]:
                    shared = set(ring_a) & set(ring_b)
                    if len(shared) > 2:
                        score += 1.0

        return min(10.0, max(1.0, score))
    except Exception:
        return None

def is_synthetically_accessible(mol_or_smiles, max_score: float = 4.5) -> bool:
    """
    Check if molecule is synthetically accessible (SA score ≤ threshold).

    Args:
        mol_or_smiles: RDKit Mol or SMILES
        max_score: Maximum acceptable SA score

    Returns:
        True if synthetically accessible
    """
    score = calculate_sa_score(mol_or_smiles)
    if score is None:
        return False
    return score <= max_score

def passes_all_filters(mol_or_smiles, check_glue: bool = True) -> bool:
    """
    Check if molecule passes all quality filters.

    Args:
        mol_or_smiles: RDKit Mol or SMILES
        check_glue: Whether to check glue-likeness

    Returns:
        True if passes all filters
    """
    if not is_drug_like(mol_or_smiles):
        return False
    if check_glue and not is_glue_like(mol_or_smiles):
        return False
    if not passes_pains_filter(mol_or_smiles):
        return False
    if not is_synthetically_accessible(mol_or_smiles):
        return False
    if has_reactive_groups(mol_or_smiles):
        return False
    return True

### 3.3 · Glue-Likeness Scoring
Score molecules 0–100 for molecular-glue similarity based on warhead motifs, scaffolds, and physicochemical properties.

In [ ]:
"""
Glue-likeness scoring for molecular glue candidates.

Scores molecules from 0 to 100 based on structural and physicochemical
features characteristic of known molecular glues (IMiDs, PROTACs warheads,
aryl-sulfonamides, etc.).
"""
from typing import Optional, List, Tuple
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

# (chemistry utils defined above)
# (filter functions defined above)

# ── SMARTS definitions ──────────────────────────────────────────────

# Warhead motifs found in clinically relevant molecular glues
WARHEAD_SMARTS = {
    "glutarimide": "[#7]1C(=O)[#6][#6]C(=O)1",
    "phthalimide": "[#7]1C(=O)c2ccccc2C1=O",
    "succinimide": "[#7]1C(=O)[#6]C(=O)1",
    "sulfonamide": "[#7]S(=O)(=O)c",
    "hydantoin": "O=C1NC(=O)NC1",
    "barbiturate": "O=C1NC(=O)NC(=O)C1",
}

# Privileged scaffolds from known glue chemotypes
PRIVILEGED_SCAFFOLD_SMARTS = {
    "isoindolinone": "O=C1NCc2ccccc12",
    "benzimidazole": "c1ccc2[nH]cnc2c1",
    "benzothiazole": "c1ccc2ncsc2c1",
    "benzoxazole": "c1ccc2ncoc2c1",
    "quinazolinone": "O=c1[nH]cnc2ccccc12",
    "indole": "c1ccc2[nH]ccc2c1",
    "dihydroquinazolinone": "O=C1NC(c2ccccc2N1)c1ccccc1",
    "phenyl_piperazine": "c1ccc(N2CCNCC2)cc1",
    "phenyl_morpholine": "c1ccc(N2CCOCC2)cc1",
    "coumarin": "O=c1ccc2ccccc2o1",
}

def has_glue_warhead(mol_or_smiles) -> Tuple[bool, List[str]]:
    """
    Check if a molecule contains known molecular-glue warhead motifs.

    Args:
        mol_or_smiles: RDKit Mol object or SMILES string.

    Returns:
        Tuple of (has_warhead, list_of_matched_warhead_names).
    """
    mol = _to_mol(mol_or_smiles)
    if mol is None:
        return False, []

    matched = []
    for name, smarts in WARHEAD_SMARTS.items():
        pattern = Chem.MolFromSmarts(smarts)
        if pattern is not None and mol.HasSubstructMatch(pattern):
            matched.append(name)
    return len(matched) > 0, matched

def check_privileged_scaffolds(mol_or_smiles) -> Tuple[bool, List[str]]:
    """
    Check if a molecule contains privileged scaffolds found in molecular glues.

    Args:
        mol_or_smiles: RDKit Mol object or SMILES string.

    Returns:
        Tuple of (has_scaffold, list_of_matched_scaffold_names).
    """
    mol = _to_mol(mol_or_smiles)
    if mol is None:
        return False, []

    matched = []
    for name, smarts in PRIVILEGED_SCAFFOLD_SMARTS.items():
        pattern = Chem.MolFromSmarts(smarts)
        if pattern is not None and mol.HasSubstructMatch(pattern):
            matched.append(name)
    return len(matched) > 0, matched

def compute_glue_likeness_score(mol_or_smiles) -> int:
    """
    Score a molecule for molecular-glue-likeness on a 0–100 scale.

    Scoring breakdown (positive contributions):
        - Has warhead motif (glutarimide/phthalimide/sulfonamide …)  : +30
        - Aromatic rings 2–3                                         : +20
        - Rigidity (≤3 rotatable bonds)                              : +15
        - Balanced polarity (HBD 1–4, HBA 3–8)                      : +15
        - Proper size (MW 250–450)                                   : +10
        - Proper lipophilicity (LogP 1.5–3.5)                        : +10

    Penalties:
        - Reactive groups              : −30
        - PAINS alerts                 : −20
        - Poor permeability (TPSA>140) : −15

    Args:
        mol_or_smiles: RDKit Mol object or SMILES string.

    Returns:
        Integer score clamped to [0, 100].

    Example::

        >>> from utils.glue_scoring import compute_glue_likeness_score
        >>> compute_glue_likeness_score("O=C1NC(=O)c2ccccc12")  # phthalimide
        45
    """
    mol = _to_mol(mol_or_smiles)
    if mol is None:
        return 0

    props = get_molecular_properties(mol)
    if props is None:
        return 0

    score = 0

    # ── Positive contributions ──────────────────────────────────────
    # Warhead motif (+30)
    has_wh, _ = has_glue_warhead(mol)
    if has_wh:
        score += 30

    # Aromatic rings 2-3 (+20), partial for 1 or 4 (+10)
    arom = props.get("num_aromatic_rings", 0)
    if 2 <= arom <= 3:
        score += 20
    elif arom == 1 or arom == 4:
        score += 10

    # Rigidity: ≤3 rotatable bonds (+15), 4-5 (+8)
    rot = props.get("rotatable_bonds", 99)
    if rot <= 3:
        score += 15
    elif rot <= 5:
        score += 8

    # Balanced polarity: HBD 1-4 and HBA 3-8 (+15)
    hbd = props.get("hbd", 0)
    hba = props.get("hba", 0)
    if 1 <= hbd <= 4 and 3 <= hba <= 8:
        score += 15
    elif 0 <= hbd <= 5 and 1 <= hba <= 10:
        score += 7  # partial credit

    # Proper size: MW 250-450 (+10)
    mw = props.get("molecular_weight", 0)
    if 250 <= mw <= 450:
        score += 10
    elif 200 <= mw <= 500:
        score += 5

    # Proper LogP: 1.5-3.5 (+10)
    logp = props.get("logp", -999)
    if 1.5 <= logp <= 3.5:
        score += 10
    elif 0.0 <= logp <= 5.0:
        score += 5

    # ── Penalties ───────────────────────────────────────────────────
    if has_reactive_groups(mol):
        score -= 30

    if not passes_pains_filter(mol):
        score -= 20

    tpsa = props.get("tpsa", 0)
    if tpsa > 140:
        score -= 15

    return max(0, min(100, score))

def rank_molecules_by_glue_score(smiles_list: List[str]) -> List[Tuple[str, int]]:
    """
    Rank a list of SMILES by glue-likeness score (descending).

    Args:
        smiles_list: List of SMILES strings.

    Returns:
        List of (smiles, score) tuples sorted by score descending.
    """
    scored = []
    for smi in smiles_list:
        s = compute_glue_likeness_score(smi)
        scored.append((smi, s))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored

# ── Private helpers ─────────────────────────────────────────────────

def _to_mol(mol_or_smiles):
    """Convert SMILES string or Mol to RDKit Mol."""
    if mol_or_smiles is None:
        return None
    if isinstance(mol_or_smiles, str):
        return Chem.MolFromSmiles(mol_or_smiles)
    if hasattr(mol_or_smiles, "GetNumAtoms"):
        return mol_or_smiles
    return None

## 4 · Data Modules
### 4.1 · Molecular Graph Representation
Convert SMILES ↔ PyTorch Geometric `Data` objects with full node & edge features.

In [ ]:
"""
Molecular graph representation and conversion utilities.
Uses PyTorch Geometric Data objects for graph neural networks.
"""
from typing import Optional, List, Tuple
import torch
from torch_geometric.data import Data
from rdkit import Chem

# (config already defined above)

class MolecularGraph:
    """
    Represents a molecule as a graph for diffusion models.

    Node features:
    - Atom type (one-hot, 10)
    - Formal charge (one-hot, 5)
    - Hybridization (one-hot, 4)
    - Is aromatic (binary)
    - Is in ring (binary)
    - Number of hydrogens (integer, 1)
    - Is conjugated (binary)

    Edge features:
    - Bond type (one-hot, 5)
    - Is aromatic (binary)
    - Is conjugated (binary)
    - Is in ring (binary)
    """

    def __init__(
        self,
        node_types: torch.Tensor,          # [N] atom type indices
        node_charges: torch.Tensor,         # [N] charge indices
        edge_index: torch.Tensor,           # [2, E] edge connectivity
        edge_types: torch.Tensor,           # [E] bond type indices
        node_hybridizations: Optional[torch.Tensor] = None,  # [N] hybridization indices
        node_aromatic: Optional[torch.Tensor] = None,        # [N] aromatic flags
        node_in_ring: Optional[torch.Tensor] = None,         # [N] ring flags
        node_num_hs: Optional[torch.Tensor] = None,          # [N] num hydrogens
        node_conjugated: Optional[torch.Tensor] = None,      # [N] conjugation flags
        edge_aromatic: Optional[torch.Tensor] = None,        # [E] aromatic flags
        edge_conjugated: Optional[torch.Tensor] = None,      # [E] conjugation flags
        edge_in_ring: Optional[torch.Tensor] = None,         # [E] ring flags
    ):
        N = len(node_types)
        E = edge_types.shape[0]
        self.node_types = node_types
        self.node_charges = node_charges
        self.edge_index = edge_index
        self.edge_types = edge_types
        self.node_hybridizations = node_hybridizations if node_hybridizations is not None else torch.zeros(N, dtype=torch.long)
        self.node_aromatic = node_aromatic if node_aromatic is not None else torch.zeros(N)
        self.node_in_ring = node_in_ring if node_in_ring is not None else torch.zeros(N)
        self.node_num_hs = node_num_hs if node_num_hs is not None else torch.zeros(N)
        self.node_conjugated = node_conjugated if node_conjugated is not None else torch.zeros(N)
        self.edge_aromatic = edge_aromatic if edge_aromatic is not None else torch.zeros(E)
        self.edge_conjugated = edge_conjugated if edge_conjugated is not None else torch.zeros(E)
        self.edge_in_ring = edge_in_ring if edge_in_ring is not None else torch.zeros(E)

    @property
    def num_nodes(self) -> int:
        return len(self.node_types)

    @property
    def num_edges(self) -> int:
        return self.edge_index.shape[1]

    def to_pyg_data(self) -> Data:
        """Convert to PyTorch Geometric Data object."""
        # One-hot encode atom types
        node_type_onehot = torch.zeros(self.num_nodes, len(ATOM_TYPES))
        node_type_onehot.scatter_(1, self.node_types.unsqueeze(1), 1)

        # One-hot encode charges
        node_charge_onehot = torch.zeros(self.num_nodes, len(CHARGES))
        node_charge_onehot.scatter_(1, self.node_charges.unsqueeze(1), 1)

        # One-hot encode hybridizations
        node_hybrid_onehot = torch.zeros(self.num_nodes, len(HYBRIDIZATIONS))
        node_hybrid_onehot.scatter_(1, self.node_hybridizations.unsqueeze(1), 1)

        # Combine node features: [atom_types(10) + charges(5) + hybrid(4) + aromatic(1) + in_ring(1) + num_hs(1) + conjugated(1)] = 23
        x = torch.cat([
            node_type_onehot,
            node_charge_onehot,
            node_hybrid_onehot,
            self.node_aromatic.unsqueeze(1).float(),
            self.node_in_ring.unsqueeze(1).float(),
            self.node_num_hs.unsqueeze(1).float(),
            self.node_conjugated.unsqueeze(1).float(),
        ], dim=1)

        # Create edge feature matrix
        edge_type_onehot = torch.zeros(self.num_edges, len(BOND_TYPES))
        edge_type_onehot.scatter_(1, self.edge_types.unsqueeze(1), 1)

        # Edge features: [bond_types(5) + aromatic(1) + conjugated(1) + in_ring(1)] = 8
        edge_attr = torch.cat([
            edge_type_onehot,
            self.edge_aromatic.unsqueeze(1).float(),
            self.edge_conjugated.unsqueeze(1).float(),
            self.edge_in_ring.unsqueeze(1).float(),
        ], dim=1)

        return Data(
            x=x,
            edge_index=self.edge_index,
            edge_attr=edge_attr,
            node_types=self.node_types,
            node_charges=self.node_charges,
            node_hybridizations=self.node_hybridizations,
            edge_types=self.edge_types,
        )

def _get_hybridization_idx(atom) -> int:
    """Map RDKit hybridization to index."""
    hyb = atom.GetHybridization()
    mapping = {
        Chem.rdchem.HybridizationType.SP: HYBRID_TO_IDX['SP'],
        Chem.rdchem.HybridizationType.SP2: HYBRID_TO_IDX['SP2'],
        Chem.rdchem.HybridizationType.SP3: HYBRID_TO_IDX['SP3'],
    }
    return mapping.get(hyb, HYBRID_TO_IDX['OTHER'])

def smiles_to_graph(smiles: str) -> Optional[Data]:
    """
    Convert a SMILES string to a PyTorch Geometric Data object.

    Args:
        smiles: SMILES string

    Returns:
        PyG Data object or None if conversion fails
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        # Add hydrogens for proper valence, then remove for graph
        mol = Chem.AddHs(mol)
        mol = Chem.RemoveHs(mol)

        num_atoms = mol.GetNumAtoms()
        if num_atoms == 0:
            return None

        # Extract node features
        node_types = []
        node_charges = []
        node_hybridizations = []
        node_aromatic = []
        node_in_ring = []
        node_num_hs = []
        node_conjugated = []

        for atom in mol.GetAtoms():
            # Atom type
            symbol = atom.GetSymbol()
            if symbol in ATOM_TO_IDX:
                node_types.append(ATOM_TO_IDX[symbol])
            else:
                node_types.append(ATOM_TO_IDX['Other'])

            # Formal charge (clamp to valid range)
            charge = atom.GetFormalCharge()
            charge = max(-2, min(2, charge))
            node_charges.append(CHARGE_TO_IDX[charge])

            # Hybridization
            node_hybridizations.append(_get_hybridization_idx(atom))

            # Boolean / numeric features
            node_aromatic.append(1 if atom.GetIsAromatic() else 0)
            node_in_ring.append(1 if atom.IsInRing() else 0)
            node_num_hs.append(atom.GetTotalNumHs())
            # Atom-level conjugation: True if any neighboring bond is conjugated
            is_conj = any(b.GetIsConjugated() for b in atom.GetBonds()) if atom.GetBonds() else False
            node_conjugated.append(1 if is_conj else 0)

        # Extract edge features
        edge_indices = []
        edge_types = []
        edge_aromatic = []
        edge_conjugated = []
        edge_in_ring = []

        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            # Get bond type
            bond_type = bond.GetBondType()
            if bond_type == Chem.BondType.SINGLE:
                bt = BOND_TO_IDX['SINGLE']
            elif bond_type == Chem.BondType.DOUBLE:
                bt = BOND_TO_IDX['DOUBLE']
            elif bond_type == Chem.BondType.TRIPLE:
                bt = BOND_TO_IDX['TRIPLE']
            elif bond_type == Chem.BondType.AROMATIC:
                bt = BOND_TO_IDX['AROMATIC']
            else:
                bt = BOND_TO_IDX['SINGLE']  # Default

            # Add edges in both directions (undirected graph)
            edge_indices.append([i, j])
            edge_indices.append([j, i])
            edge_types.extend([bt, bt])

            is_aromatic = 1 if bond.GetIsAromatic() else 0
            is_conjugated = 1 if bond.GetIsConjugated() else 0
            is_in_ring = 1 if bond.IsInRing() else 0
            edge_aromatic.extend([is_aromatic, is_aromatic])
            edge_conjugated.extend([is_conjugated, is_conjugated])
            edge_in_ring.extend([is_in_ring, is_in_ring])

        # Handle molecules with no bonds (single atoms)
        if len(edge_indices) == 0:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_types_t = torch.zeros(0, dtype=torch.long)
            edge_aromatic_t = torch.zeros(0)
            edge_conjugated_t = torch.zeros(0)
            edge_in_ring_t = torch.zeros(0)
        else:
            edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
            edge_types_t = torch.tensor(edge_types, dtype=torch.long)
            edge_aromatic_t = torch.tensor(edge_aromatic, dtype=torch.float)
            edge_conjugated_t = torch.tensor(edge_conjugated, dtype=torch.float)
            edge_in_ring_t = torch.tensor(edge_in_ring, dtype=torch.float)

        # Create MolecularGraph
        mol_graph = MolecularGraph(
            node_types=torch.tensor(node_types, dtype=torch.long),
            node_charges=torch.tensor(node_charges, dtype=torch.long),
            edge_index=edge_index,
            edge_types=edge_types_t,
            node_hybridizations=torch.tensor(node_hybridizations, dtype=torch.long),
            node_aromatic=torch.tensor(node_aromatic, dtype=torch.float),
            node_in_ring=torch.tensor(node_in_ring, dtype=torch.float),
            node_num_hs=torch.tensor(node_num_hs, dtype=torch.float),
            node_conjugated=torch.tensor(node_conjugated, dtype=torch.float),
            edge_aromatic=edge_aromatic_t,
            edge_conjugated=edge_conjugated_t,
            edge_in_ring=edge_in_ring_t,
        )

        # Convert to PyG Data
        data = mol_graph.to_pyg_data()
        data.smiles = smiles

        return data

    except Exception as e:
        return None

def graph_to_smiles(data: Data) -> Optional[str]:
    """
    Convert a PyTorch Geometric Data object back to SMILES.

    Uses a connectivity-maximizing strategy: builds a spanning tree first
    to ensure all atoms are connected, then adds remaining bonds greedily
    while respecting valence limits. This avoids the fragmentation problem
    where greedy pruning produces tiny disconnected pieces.

    Args:
        data: PyG Data object with node_types and edge_types

    Returns:
        SMILES string or None if conversion fails
    """
    try:
        # Get node and edge types
        if hasattr(data, 'node_types'):
            node_types = data.node_types.cpu().numpy()
        else:
            node_types = data.x[:, :len(ATOM_TYPES)].argmax(dim=1).cpu().numpy()

        if hasattr(data, 'edge_types'):
            edge_types = data.edge_types.cpu().numpy()
        else:
            edge_types = data.edge_attr[:, :len(BOND_TYPES)].argmax(dim=1).cpu().numpy()

        edge_index = data.edge_index.cpu().numpy()

        # ---- Collect bond list (deduplicated, skip NONE) ----
        bonds = {}
        for idx in range(edge_index.shape[1]):
            i, j = int(edge_index[0, idx]), int(edge_index[1, idx])
            if i >= j:
                continue
            bt_idx = int(edge_types[idx])
            bt_str = BOND_TYPES[bt_idx]
            if bt_str == 'NONE':
                continue
            key = (i, j)
            if key not in bonds:
                bonds[key] = bt_str

        if not bonds:
            return None

        # ---- Map atom types ----
        # Expanded max valences (allow N=5, S=6 for charged/oxidized forms)
        MAX_VALENCE = {'C': 4, 'N': 3, 'O': 2, 'S': 6, 'F': 1,
                       'Cl': 1, 'Br': 1, 'P': 5, 'I': 1}
        BOND_ORDER = {'SINGLE': 1, 'DOUBLE': 2, 'TRIPLE': 3, 'AROMATIC': 1.5}

        symbols = []
        for at_idx in node_types:
            sym = ATOM_TYPES[at_idx]
            if sym == 'Other':
                sym = 'C'
            symbols.append(sym)

        num_atoms = len(symbols)

        # Convert AROMATIC bonds to SINGLE for stability, TRIPLE→DOUBLE for safety
        clean_bonds = {}
        for (i, j), bt in bonds.items():
            if bt == 'AROMATIC':
                bt = 'SINGLE'
            elif bt == 'TRIPLE':
                bt = 'DOUBLE'
            clean_bonds[(i, j)] = bt

        # ---- PHASE 1: Build spanning tree to maximise connectivity ----
        # Scoring: prefer bonds involving carbon (higher-valence, backbone atoms)
        def bond_score(key):
            i, j = key
            bt = clean_bonds[key]
            score = 0
            # Strongly prefer C–C bonds (scaffold backbone)
            if symbols[i] == 'C' and symbols[j] == 'C':
                score += 100
            # Prefer bonds involving at least one carbon
            elif symbols[i] == 'C' or symbols[j] == 'C':
                score += 50
            # Prefer SINGLE bonds (use less valence budget)
            if bt == 'SINGLE':
                score += 10
            return score

        sorted_bond_keys = sorted(clean_bonds.keys(), key=bond_score, reverse=True)

        # Union-Find for spanning tree
        parent = list(range(num_atoms))
        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x
        def union(a, b):
            ra, rb = find(a), find(b)
            if ra == rb:
                return False
            parent[ra] = rb
            return True

        atom_valence_used = [0.0] * num_atoms
        accepted_bonds = []

        # First pass: spanning tree edges (connect disconnected components)
        remaining = []
        for key in sorted_bond_keys:
            i, j = key
            bt = clean_bonds[key]
            order = BOND_ORDER.get(bt, 1)
            max_i = MAX_VALENCE.get(symbols[i], 4)
            max_j = MAX_VALENCE.get(symbols[j], 4)

            if atom_valence_used[i] + order <= max_i and atom_valence_used[j] + order <= max_j:
                if union(i, j):
                    accepted_bonds.append((key, bt))
                    atom_valence_used[i] += order
                    atom_valence_used[j] += order
                else:
                    remaining.append(key)
            else:
                # Try downgrading to SINGLE if it was DOUBLE
                if bt == 'DOUBLE' and atom_valence_used[i] + 1 <= max_i and atom_valence_used[j] + 1 <= max_j:
                    if union(i, j):
                        accepted_bonds.append((key, 'SINGLE'))
                        atom_valence_used[i] += 1
                        atom_valence_used[j] += 1
                    else:
                        remaining.append(key)

        # ---- PHASE 2: Add ring-closing / extra bonds from remaining ----
        for key in remaining:
            i, j = key
            bt = clean_bonds[key]
            order = BOND_ORDER.get(bt, 1)
            max_i = MAX_VALENCE.get(symbols[i], 4)
            max_j = MAX_VALENCE.get(symbols[j], 4)

            if atom_valence_used[i] + order <= max_i and atom_valence_used[j] + order <= max_j:
                accepted_bonds.append((key, bt))
                atom_valence_used[i] += order
                atom_valence_used[j] += order
            elif bt == 'DOUBLE' and atom_valence_used[i] + 1 <= max_i and atom_valence_used[j] + 1 <= max_j:
                accepted_bonds.append((key, 'SINGLE'))
                atom_valence_used[i] += 1
                atom_valence_used[j] += 1

        # ---- Identify connected atoms and remove isolates ----
        connected = set()
        for (i, j), _ in accepted_bonds:
            connected.add(i)
            connected.add(j)

        if len(connected) < 3:
            return None

        # Reindex to remove isolated atoms
        old_to_new = {}
        new_symbols = []
        for old_idx in sorted(connected):
            old_to_new[old_idx] = len(new_symbols)
            new_symbols.append(symbols[old_idx])

        # ---- Build RDKit molecule ----
        mol = Chem.RWMol()
        for sym in new_symbols:
            mol.AddAtom(Chem.Atom(sym))

        RDKIT_BOND = {
            'SINGLE': Chem.BondType.SINGLE,
            'DOUBLE': Chem.BondType.DOUBLE,
            'TRIPLE': Chem.BondType.TRIPLE,
        }
        for (i, j), bt in accepted_bonds:
            if i in old_to_new and j in old_to_new:
                ni, nj = old_to_new[i], old_to_new[j]
                mol.AddBond(ni, nj, RDKIT_BOND.get(bt, Chem.BondType.SINGLE))

        # ---- Sanitize and extract SMILES ----
        try:
            Chem.SanitizeMol(mol)
            smiles = Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            try:
                Chem.SanitizeMol(mol, Chem.SanitizeFlags.SANITIZE_FINDRADICALS |
                                 Chem.SanitizeFlags.SANITIZE_SETAROMATICITY |
                                 Chem.SanitizeFlags.SANITIZE_SETCONJUGATION |
                                 Chem.SanitizeFlags.SANITIZE_SETHYBRIDIZATION |
                                 Chem.SanitizeFlags.SANITIZE_SYMMRINGS)
                smiles = Chem.MolToSmiles(mol, canonical=False)
            except Exception:
                try:
                    smiles = Chem.MolToSmiles(mol, canonical=False)
                except Exception:
                    return None

        if not smiles:
            return None

        # Take largest fragment if disconnected
        if '.' in smiles:
            fragments = smiles.split('.')
            smiles = max(fragments, key=len)

        # Final validity check
        check_mol = Chem.MolFromSmiles(smiles)
        if check_mol is None:
            return None
        if check_mol.GetNumHeavyAtoms() < 3:
            return None

        return Chem.MolToSmiles(check_mol, canonical=True)

    except Exception:
        return None

def batch_smiles_to_graphs(smiles_list: List[str]) -> List[Data]:
    """
    Convert a list of SMILES to graphs, filtering invalid ones.

    Args:
        smiles_list: List of SMILES strings

    Returns:
        List of valid PyG Data objects
    """
    graphs = []
    for smiles in smiles_list:
        graph = smiles_to_graph(smiles)
        if graph is not None:
            graphs.append(graph)
    return graphs

### 4.2 · Dataset & Data Loading
PyTorch Geometric `InMemoryDataset` that loads molecules from CSV/SMILES files and converts them to graphs.

In [ ]:
"""
Dataset class for molecular glue training data.
"""
import os
from typing import Optional, List, Callable
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from tqdm import tqdm

from .molecular_graph import smiles_to_graph

# (filter functions defined above)
# (chemistry utils defined above)
# (config already defined above)

def _extract_property_tensor(smiles: str) -> Optional[torch.Tensor]:
    """
    Compute property annotations for conditioning.

    Returns:
        Tensor of shape [num_properties] with normalized property values,
        or None if properties can't be computed.
    """
    props = get_molecular_properties(smiles)
    if props is None:
        return None

    # SA score helper
    from utils.filters import calculate_sa_score
    sa = calculate_sa_score(smiles)
    if sa is None:
        sa = 3.0  # default

    # Build tensor in PROPERTY_NAMES order
    values = [
        props.get('molecular_weight', 300.0) / 500.0,  # normalize to ~[0, 1]
        (props.get('logp', 2.0) + 2.0) / 8.0,          # shift & scale
        props.get('num_aromatic_rings', 1) / 5.0,
        props.get('hbd', 1) / 5.0,
        props.get('hba', 3) / 10.0,
        props.get('fraction_sp3', 0.3),                 # already [0, 1]
        props.get('tpsa', 60.0) / 140.0,
        sa / 10.0,
    ]
    return torch.tensor(values, dtype=torch.float32)

class MolecularGlueDataset(Dataset):
    """
    PyTorch Dataset for molecular glue-like molecules.

    Loads molecules from CSV/SMILES files, converts to graphs,
    and filters by drug-likeness and glue-like properties.
    Each graph item includes a `properties` tensor for conditioning.
    """

    def __init__(
        self,
        data_path: str,
        smiles_column: str = 'smiles',
        transform: Optional[Callable] = None,
        filter_drug_like: bool = True,
        filter_glue_like: bool = False,
        cache_graphs: bool = True,
        max_atoms: int = 50,
        verbose: bool = True,
    ):
        """
        Args:
            data_path: Path to CSV file or directory of SMILES files
            smiles_column: Column name for SMILES in CSV
            transform: Optional transform to apply to graphs
            filter_drug_like: Filter by Lipinski's Rule of Five
            filter_glue_like: Filter by glue-like properties
            cache_graphs: Cache converted graphs in memory
            max_atoms: Maximum number of atoms (filter larger molecules)
            verbose: Print loading progress
        """
        self.data_path = data_path
        self.smiles_column = smiles_column
        self.transform = transform
        self.filter_drug_like = filter_drug_like
        self.filter_glue_like = filter_glue_like
        self.cache_graphs = cache_graphs
        self.max_atoms = max_atoms
        self.verbose = verbose

        # Cache for converted graphs
        self._graph_cache = {}

        # Load and filter SMILES
        self.smiles_list = self._load_smiles()

        if verbose:
            print(f"Loaded {len(self.smiles_list)} molecules from {data_path}")

    def _load_smiles(self) -> List[str]:
        """Load SMILES from file(s) and apply filters."""
        smiles_list = []

        if os.path.isfile(self.data_path):
            if self.data_path.endswith('.csv'):
                df = pd.read_csv(self.data_path)
                smiles_list = df[self.smiles_column].dropna().tolist()
            elif self.data_path.endswith('.smi') or self.data_path.endswith('.txt'):
                with open(self.data_path, 'r') as f:
                    smiles_list = [line.strip().split()[0] for line in f if line.strip()]
        elif os.path.isdir(self.data_path):
            # Load from directory of files
            for filename in os.listdir(self.data_path):
                filepath = os.path.join(self.data_path, filename)
                if filename.endswith('.csv'):
                    df = pd.read_csv(filepath)
                    smiles_list.extend(df[self.smiles_column].dropna().tolist())
                elif filename.endswith('.smi') or filename.endswith('.txt'):
                    with open(filepath, 'r') as f:
                        smiles_list.extend([line.strip().split()[0] for line in f if line.strip()])

        # Filter molecules
        filtered_smiles = []
        iterator = tqdm(smiles_list, desc="Filtering molecules") if self.verbose else smiles_list

        for smiles in iterator:
            # Convert to graph to check validity and size
            graph = smiles_to_graph(smiles)
            if graph is None:
                continue

            if graph.x.shape[0] > self.max_atoms:
                continue

            # Drug-likeness filter
            if self.filter_drug_like and not is_drug_like(smiles):
                continue

            # Glue-like filter (optional, more restrictive)
            if self.filter_glue_like and not is_glue_like(smiles):
                continue

            # Compute property annotations for conditioning
            prop_tensor = _extract_property_tensor(smiles)
            if prop_tensor is not None:
                graph.properties = prop_tensor

            filtered_smiles.append(smiles)

            # Cache the graph if enabled
            if self.cache_graphs:
                self._graph_cache[len(filtered_smiles) - 1] = graph

        return filtered_smiles

    def __len__(self) -> int:
        return len(self.smiles_list)

    def __getitem__(self, idx: int) -> Data:
        """Get graph for molecule at index."""
        # Check cache first
        if self.cache_graphs and idx in self._graph_cache:
            graph = self._graph_cache[idx]
        else:
            smiles = self.smiles_list[idx]
            graph = smiles_to_graph(smiles)
            if graph is None:
                # Return a dummy graph if conversion fails
                graph = Data(
                    x=torch.zeros((1, 23)),  # Updated node feature dim
                    edge_index=torch.zeros((2, 0), dtype=torch.long),
                    edge_attr=torch.zeros((0, 8)),  # Updated edge feature dim
                    properties=torch.zeros(len(PROPERTY_NAMES)),
                )
            else:
                # Compute properties if not cached
                prop_tensor = _extract_property_tensor(smiles)
                if prop_tensor is not None:
                    graph.properties = prop_tensor
                else:
                    graph.properties = torch.zeros(len(PROPERTY_NAMES))

            if self.cache_graphs:
                self._graph_cache[idx] = graph

        if self.transform:
            graph = self.transform(graph)

        return graph

    def get_smiles(self, idx: int) -> str:
        """Get SMILES string at index."""
        return self.smiles_list[idx]

    def get_all_smiles(self) -> List[str]:
        """Get all SMILES strings."""
        return self.smiles_list.copy()

    @staticmethod
    def collate_fn(batch: List[Data]):
        """Custom collate function for batching graphs."""
        from torch_geometric.data import Batch
        return Batch.from_data_list(batch)

def create_train_val_split(
    dataset: MolecularGlueDataset,
    val_ratio: float = 0.1,
    seed: int = 42,
) -> tuple:
    """
    Split dataset into train and validation sets.

    Args:
        dataset: MolecularGlueDataset instance
        val_ratio: Fraction of data for validation
        seed: Random seed

    Returns:
        Tuple of (train_dataset, val_dataset)
    """
    from torch.utils.data import Subset
    import numpy as np

    np.random.seed(seed)
    indices = np.random.permutation(len(dataset))

    val_size = int(len(dataset) * val_ratio)
    val_indices = indices[:val_size]
    train_indices = indices[val_size:]

    train_dataset = Subset(dataset, train_indices.tolist())
    val_dataset = Subset(dataset, val_indices.tolist())

    return train_dataset, val_dataset

### 4.3 · Data Download & Preprocessing
Functions to download training molecules from ChEMBL and preprocess/merge datasets.

In [ ]:
"""
Download drug-like molecules from ChEMBL and merge with existing glue chemotypes.

Usage:
    pip install chembl_webresource_client   # if not already installed
    python scripts/download_training_data.py --output data/training_data.csv

If ChEMBL is unavailable, a ZINC15 download fallback is provided.
"""
import argparse
import os
import sys

def download_chembl_druglike(n_molecules: int = 50_000) -> list:
    """Download drug-like molecules from ChEMBL."""
    try:
        from chembl_webresource_client.new_client import new_client
    except ImportError:
        print("ERROR: chembl_webresource_client not installed.")
        print("Install with: pip install chembl_webresource_client")
        return []

    molecule = new_client.molecule
    print(f"Querying ChEMBL for up to {n_molecules} drug-like molecules …")

    results = molecule.filter(
        molecule_properties__mw_freebase__lte=500,
        molecule_properties__mw_freebase__gte=200,
        molecule_properties__alogp__lte=5,
        molecule_properties__hbd__lte=5,
        molecule_properties__hba__lte=10,
        molecule_properties__aromatic_rings__gte=1,
    ).only("molecule_structures")

    smiles_list = []
    for rec in results:
        if len(smiles_list) >= n_molecules:
            break
        struct = rec.get("molecule_structures")
        if struct and struct.get("canonical_smiles"):
            smiles_list.append(struct["canonical_smiles"])

        if len(smiles_list) % 5000 == 0 and len(smiles_list) > 0:
            print(f"  … downloaded {len(smiles_list)} so far")

    print(f"Downloaded {len(smiles_list)} molecules from ChEMBL")
    return smiles_list

def validate_smiles_list(smiles_list: list) -> list:
    """Validate SMILES using RDKit."""
    from rdkit import Chem

    valid = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            try:
                Chem.SanitizeMol(mol)
                canonical = Chem.MolToSmiles(mol, canonical=True)
                valid.append(canonical)
            except Exception:
                pass

    print(f"Validated: {len(valid)}/{len(smiles_list)} SMILES are valid")
    return valid

def merge_datasets(
    chembl_smiles: list,
    existing_csv: str,
    output_csv: str,
    glue_fraction: float = 0.15,
):
    """Merge ChEMBL molecules with existing glue chemotypes."""
    import pandas as pd

    # Load existing glues
    if os.path.exists(existing_csv):
        df_existing = pd.read_csv(existing_csv)
        existing_smiles = df_existing["smiles"].dropna().tolist()
        print(f"Existing glue chemotypes: {len(existing_smiles)}")
    else:
        existing_smiles = []
        print(f"No existing file at {existing_csv}")

    # Deduplicate
    all_smiles = list(set(chembl_smiles + existing_smiles))

    # Ensure glues are represented at desired fraction via oversampling
    if existing_smiles and glue_fraction > 0:
        target_glue_count = int(len(all_smiles) * glue_fraction)
        oversample = max(1, target_glue_count // max(len(existing_smiles), 1))
        oversampled_glues = existing_smiles * oversample
        all_smiles = list(set(all_smiles + oversampled_glues[: target_glue_count]))

    df = pd.DataFrame({"smiles": all_smiles})
    os.makedirs(os.path.dirname(output_csv) if os.path.dirname(output_csv) else ".", exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(all_smiles)} molecules to {output_csv}")

def main():
    parser = argparse.ArgumentParser(description="Download & merge training data")
    parser.add_argument("--n_molecules", type=int, default=50_000)
    parser.add_argument("--existing", type=str, default="data/glue_chemotypes.csv")
    parser.add_argument("--output", type=str, default="data/training_data.csv")
    parser.add_argument("--glue_fraction", type=float, default=0.15,
                        help="Fraction of final dataset that should be glue chemotypes")
    args = parser.parse_args()

    # Download
    smiles = download_chembl_druglike(args.n_molecules)
    if not smiles:
        print("\nChEMBL download failed. Please install chembl_webresource_client "
              "or manually provide drug-like SMILES in data/training_data.csv")
        return

    # Validate
    smiles = validate_smiles_list(smiles)

    # Merge
    merge_datasets(smiles, args.existing, args.output, args.glue_fraction)

"""
Preprocess and merge datasets for training the molecular diffusion model.

Merges ChEMBL drug-like molecules with known glue chemotypes, deduplicates,
balances the glue fraction via oversampling, creates train/val splits, and
saves dataset statistics.

Usage:
    python scripts/preprocess_data.py \\
        --chembl data/raw/chembl_druglike.csv \\
        --glues data/glue_chemotypes.csv \\
        --output data/processed/training_data.csv \\
        --glue_fraction 0.15 \\
        --val_split 0.1
"""
import argparse
import json
import os
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

from rdkit import Chem
# (chemistry utils defined above)

def load_smiles_from_csv(path: str, smiles_col: str = "smiles") -> pd.DataFrame:
    """
    Load and validate SMILES from a CSV file.

    Args:
        path: Path to the CSV.
        smiles_col: Column containing SMILES strings.

    Returns:
        DataFrame with at least a ``smiles`` column of canonical SMILES.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(path)
    if smiles_col not in df.columns:
        # Try common alternatives
        for alt in ("SMILES", "Smiles", "canonical_smiles"):
            if alt in df.columns:
                smiles_col = alt
                break
        else:
            raise KeyError(
                f"Column '{smiles_col}' not found. Available: {list(df.columns)}"
            )

    raw = df[smiles_col].dropna().tolist()
    canonical = []
    for smi in raw:
        c = canonicalize_smiles(str(smi))
        if c and is_valid_molecule(c):
            canonical.append(c)

    print(f"  Loaded {len(canonical)}/{len(raw)} valid SMILES from {path}")
    return pd.DataFrame({"smiles": canonical})

def compute_all_properties(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute molecular properties for each SMILES and add as columns.

    Args:
        df: DataFrame with ``smiles`` column.

    Returns:
        DataFrame augmented with property columns.
    """
    records = []
    for smi in tqdm(df["smiles"], desc="Computing properties", unit="mol"):
        props = get_molecular_properties(smi)
        if props is None:
            props = {}
        props["smiles"] = smi
        records.append(props)
    return pd.DataFrame(records)

def merge_and_balance(
    df_chembl: pd.DataFrame,
    df_glues: pd.DataFrame,
    glue_fraction: float = 0.15,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Merge ChEMBL and glue datasets, deduplicate, and balance via oversampling.

    Args:
        df_chembl: ChEMBL drug-like molecules (must have ``smiles``).
        df_glues: Known glue chemotypes (must have ``smiles``).
        glue_fraction: Target fraction of glue molecules in final dataset.
        seed: Random seed for shuffling.

    Returns:
        Merged, shuffled DataFrame with ``is_glue`` column.
    """
    # Tag sources
    df_chembl = df_chembl.copy()
    df_glues = df_glues.copy()
    df_chembl["is_glue"] = 0
    df_glues["is_glue"] = 1

    # Deduplicate within each set
    df_chembl = df_chembl.drop_duplicates(subset="smiles")
    df_glues = df_glues.drop_duplicates(subset="smiles")

    # Remove glues that happen to be in ChEMBL set already
    glue_set = set(df_glues["smiles"])
    df_chembl = df_chembl[~df_chembl["smiles"].isin(glue_set)]

    n_chembl = len(df_chembl)
    n_glues_orig = len(df_glues)

    # Compute how many glue copies are needed
    # Want: n_glue_final / (n_chembl + n_glue_final) = glue_fraction
    # => n_glue_final = glue_fraction * n_chembl / (1 - glue_fraction)
    if glue_fraction > 0 and n_glues_orig > 0:
        n_glue_target = int(glue_fraction * n_chembl / (1 - glue_fraction))
        n_glue_target = max(n_glue_target, n_glues_orig)  # at least original count

        if n_glue_target > n_glues_orig:
            # Oversample glues
            rng = np.random.RandomState(seed)
            repeats = n_glue_target // n_glues_orig
            remainder = n_glue_target % n_glues_orig
            parts = [df_glues] * repeats
            if remainder > 0:
                parts.append(df_glues.sample(n=remainder, random_state=rng))
            df_glues = pd.concat(parts, ignore_index=True)

        print(f"  Oversampled glues: {n_glues_orig} → {len(df_glues)} "
              f"(target fraction: {glue_fraction:.0%})")

    # Merge
    df_merged = pd.concat([df_chembl, df_glues], ignore_index=True)
    df_merged = df_merged.drop_duplicates(subset="smiles")

    # Shuffle
    df_merged = df_merged.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    actual_frac = df_merged["is_glue"].mean()
    print(f"  Final dataset: {len(df_merged)} molecules "
          f"({actual_frac:.1%} glues)")

    return df_merged

def compute_dataset_stats(df: pd.DataFrame) -> dict:
    """
    Compute summary statistics for the dataset.

    Args:
        df: DataFrame with property columns.

    Returns:
        Dictionary of statistics suitable for JSON serialisation.
    """
    props = [
        "molecular_weight", "logp", "hbd", "hba", "tpsa",
        "rotatable_bonds", "num_aromatic_rings", "num_rings",
        "num_heavy_atoms", "fraction_sp3", "qed",
    ]
    stats: dict = {"n_molecules": len(df)}

    for p in props:
        if p not in df.columns:
            continue
        vals = df[p].dropna()
        stats[p] = {
            "mean": round(float(vals.mean()), 3),
            "std": round(float(vals.std()), 3),
            "min": round(float(vals.min()), 3),
            "max": round(float(vals.max()), 3),
            "median": round(float(vals.median()), 3),
        }

    if "is_glue" in df.columns:
        stats["n_glues"] = int(df["is_glue"].sum())
        stats["glue_fraction"] = round(float(df["is_glue"].mean()), 4)

    return stats

def train_val_split(
    df: pd.DataFrame,
    val_split: float = 0.1,
    seed: int = 42,
) -> tuple:
    """
    Split DataFrame into train and validation sets.

    Args:
        df: Full dataset.
        val_split: Fraction for validation.
        seed: Random seed.

    Returns:
        (df_train, df_val) tuple.
    """
    n_val = max(1, int(len(df) * val_split))
    rng = np.random.RandomState(seed)
    indices = rng.permutation(len(df))
    val_idx = indices[:n_val]
    train_idx = indices[n_val:]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[val_idx].reset_index(drop=True)

# ── CLI ─────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(
        description="Preprocess and merge training datasets"
    )
    parser.add_argument(
        "--chembl", type=str, default="data/raw/chembl_druglike.csv",
        help="Path to ChEMBL drug-like CSV",
    )
    parser.add_argument(
        "--glues", type=str, default="data/glue_chemotypes.csv",
        help="Path to glue chemotypes CSV",
    )
    parser.add_argument(
        "--output", type=str, default="data/processed/training_data.csv",
        help="Output path for merged dataset",
    )
    parser.add_argument(
        "--glue_fraction", type=float, default=0.15,
        help="Target fraction of glue molecules (default: 0.15)",
    )
    parser.add_argument(
        "--val_split", type=float, default=0.1,
        help="Fraction for validation split (default: 0.1)",
    )
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--force", action="store_true",
        help="Overwrite existing output",
    )
    args = parser.parse_args()

    if os.path.exists(args.output) and not args.force:
        print(f"Output file already exists: {args.output}")
        print("Use --force to overwrite.")
        return

    # ── Load data ───────────────────────────────────────────────────
    print("Loading ChEMBL data …")
    df_chembl = load_smiles_from_csv(args.chembl)

    print("Loading glue chemotypes …")
    df_glues = load_smiles_from_csv(args.glues)

    # ── Merge and balance ───────────────────────────────────────────
    print("Merging and balancing …")
    df_merged = merge_and_balance(
        df_chembl, df_glues,
        glue_fraction=args.glue_fraction,
        seed=args.seed,
    )

    # ── Compute properties ──────────────────────────────────────────
    print("Computing molecular properties …")
    df_props = compute_all_properties(df_merged)

    # Keep is_glue column
    if "is_glue" in df_merged.columns:
        df_props["is_glue"] = df_merged["is_glue"].values[
            : len(df_props)
        ]

    # ── Train/val split ─────────────────────────────────────────────
    df_train, df_val = train_val_split(df_props, val_split=args.val_split, seed=args.seed)
    print(f"Train: {len(df_train)} molecules, Val: {len(df_val)} molecules")

    # ── Save ────────────────────────────────────────────────────────
    out_dir = os.path.dirname(args.output) or "."
    os.makedirs(out_dir, exist_ok=True)

    # Full dataset
    df_props.to_csv(args.output, index=False)
    print(f"Saved full dataset to {args.output}")

    # Train/val splits
    train_path = args.output.replace(".csv", "_train.csv")
    val_path = args.output.replace(".csv", "_val.csv")
    df_train.to_csv(train_path, index=False)
    df_val.to_csv(val_path, index=False)
    print(f"Saved train split to {train_path}")
    print(f"Saved val split to {val_path}")

    # Statistics
    stats = compute_dataset_stats(df_props)
    stats_path = os.path.join(out_dir, "dataset_stats.json")
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)
    print(f"Saved dataset statistics to {stats_path}")

    # Print summary
    print(f"\n{'='*50}")
    print(f"PREPROCESSING COMPLETE")
    print(f"{'='*50}")
    print(f"Total molecules: {stats['n_molecules']}")
    if "n_glues" in stats:
        print(f"Glue molecules:  {stats['n_glues']} ({stats['glue_fraction']:.1%})")
    for prop in ["molecular_weight", "logp", "hbd", "hba", "tpsa"]:
        if prop in stats:
            s = stats[prop]
            print(f"  {prop:20s}: {s['mean']:.2f} ± {s['std']:.2f} "
                  f"[{s['min']:.1f} – {s['max']:.1f}]")

## 5 · Model Components
### 5.1 · Noise Scheduler
Discrete diffusion noise scheduler with forward (noise-adding) and reverse (posterior) processes.

In [ ]:
"""
Discrete diffusion noise scheduler for molecular graphs.

Implements D3PM-style discrete diffusion for categorical features
(atom types, bond types) rather than continuous diffusion.
"""
import math
from typing import Tuple, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F

# (config already defined above)

class NoiseScheduler:
    """
    Discrete diffusion noise scheduler.

    For categorical data, the forward process gradually corrupts
    categories towards a uniform distribution over classes.
    """

    def __init__(self, config: DiffusionConfig, device: str = 'cpu'):
        """
        Args:
            config: Diffusion configuration
            device: Device to place tensors on
        """
        self.config = config
        self.num_timesteps = config.num_timesteps
        self.device = device

        # Number of categories
        self.num_atom_types = len(ATOM_TYPES)
        self.num_bond_types = len(BOND_TYPES)

        # Create beta schedule
        self.betas = self._create_beta_schedule().to(device)

        # Cumulative products for diffusion
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        # For discrete diffusion: transition matrices
        # Q_t[i,j] = P(x_t = j | x_{t-1} = i)
        self._precompute_transition_matrices()

    def _create_beta_schedule(self) -> torch.Tensor:
        """Create noise schedule (linear or cosine)."""
        if self.config.beta_schedule == "linear":
            return torch.linspace(
                self.config.beta_start,
                self.config.beta_end,
                self.num_timesteps,
                dtype=torch.float32
            )
        elif self.config.beta_schedule == "cosine":
            # Cosine schedule from "Improved Denoising Diffusion"
            steps = self.num_timesteps + 1
            x = torch.linspace(0, self.num_timesteps, steps, dtype=torch.float32)
            alphas_cumprod = torch.cos(((x / self.num_timesteps) + 0.008) / 1.008 * math.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            return torch.clamp(betas, 0.0001, 0.9999)
        else:
            raise ValueError(f"Unknown beta schedule: {self.config.beta_schedule}")

    def _precompute_transition_matrices(self):
        """
        Precompute transition matrices for discrete diffusion.

        For absorbing state diffusion, we transition towards a uniform distribution.
        Q_t = (1 - beta_t) * I + beta_t * uniform
        """
        # Cumulative transition matrices: Q_bar_t = Q_1 @ Q_2 @ ... @ Q_t
        # For uniform noise: Q_bar_t[i,j] = alpha_bar_t * I[i,j] + (1 - alpha_bar_t) / K
        self.alpha_bar = self.alphas_cumprod

    def get_transition_probs(
        self,
        t: torch.Tensor,
        num_classes: int,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Get transition probabilities for timestep t.

        Args:
            t: Timesteps [B]
            num_classes: Number of categories (K)

        Returns:
            alpha_bar: Probability of staying in original class [B]
            uniform_prob: Probability mass for each other class [B]
        """
        alpha_bar = self.alpha_bar[t]  # [B]
        uniform_prob = (1.0 - alpha_bar) / num_classes  # [B]

        return alpha_bar, uniform_prob

    def add_noise(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        num_classes: int,
    ) -> torch.Tensor:
        """
        Add noise to categorical data at timestep t.

        Forward process: q(x_t | x_0)

        Args:
            x: Original categorical data [N] or [N, 1] (class indices)
            t: Timesteps [B] (one per batch, expanded to match x)
            num_classes: Number of categories

        Returns:
            Noisy categorical data [N] (class indices)
        """
        x = x.flatten()

        # Get transition probabilities
        alpha_bar, uniform_prob = self.get_transition_probs(t, num_classes)

        # Expand to match x shape
        if alpha_bar.dim() == 0:
            alpha_bar = alpha_bar.unsqueeze(0)
            uniform_prob = uniform_prob.unsqueeze(0)

        # Repeat for each element in x
        alpha_bar = alpha_bar.repeat_interleave(x.shape[0] // t.shape[0])
        uniform_prob = uniform_prob.repeat_interleave(x.shape[0] // t.shape[0])

        # Build transition probabilities for each element
        # P(x_t = k | x_0 = j) = alpha_bar * I[k==j] + (1 - alpha_bar) / K
        probs = torch.zeros(x.shape[0], num_classes, device=x.device)
        probs.scatter_(1, x.unsqueeze(1), 1.0)  # One-hot of original

        probs = alpha_bar.unsqueeze(1) * probs + uniform_prob.unsqueeze(1)

        # Sample from categorical distribution
        noisy_x = torch.multinomial(probs, 1).squeeze(1)

        return noisy_x

    def get_posterior_probs(
        self,
        x_t: torch.Tensor,
        x_0_pred: torch.Tensor,
        t: torch.Tensor,
        num_classes: int,
    ) -> torch.Tensor:
        """
        Compute posterior q(x_{t-1} | x_t, x_0) for reverse process.

        Args:
            x_t: Noisy data at timestep t [N]
            x_0_pred: Predicted clean data [N, K] (logits)
            t: Current timestep [B]
            num_classes: Number of categories

        Returns:
            Posterior probabilities [N, K]
        """
        # Convert x_0_pred logits to probabilities
        p_x0 = F.softmax(x_0_pred, dim=-1)  # [N, K]

        # Get alpha values
        alpha_t = self.alphas[t]  # [B]
        alpha_bar_t = self.alphas_cumprod[t]  # [B]
        alpha_bar_t_minus_1 = self.alphas_cumprod_prev[t]  # [B]

        # Expand to match N
        N = x_t.shape[0]
        B = t.shape[0]
        alpha_t = alpha_t.repeat_interleave(N // B)
        alpha_bar_t = alpha_bar_t.repeat_interleave(N // B)
        alpha_bar_t_minus_1 = alpha_bar_t_minus_1.repeat_interleave(N // B)

        # q(x_{t-1} | x_t, x_0) ∝ q(x_t | x_{t-1}) * q(x_{t-1} | x_0)
        # This is a complex formula for discrete diffusion
        # Simplified: use predicted x_0 distribution directly for sampling

        # Mix with uniform for numerical stability
        uniform = torch.ones_like(p_x0) / num_classes
        posterior = (1.0 - 0.001) * p_x0 + 0.001 * uniform

        return posterior

    def sample_timesteps(self, batch_size: int) -> torch.Tensor:
        """Sample random timesteps for training."""
        return torch.randint(0, self.num_timesteps, (batch_size,), device=self.device)

### 5.2 · Fragment Vocabulary
BRICS-based fragment vocabulary for scoring generated molecules.

In [ ]:
"""
Fragment vocabulary for fragment frequency loss.

Pre-computes fragment frequencies from training data using BRICS decomposition
and provides scoring functions for generated molecules.
"""
import json
import os
from typing import Dict, List, Optional
from collections import Counter
import math

from rdkit import Chem
from rdkit.Chem import BRICS, AllChem

class FragmentVocab:
    """
    Fragment frequency vocabulary from training data.

    Uses BRICS decomposition to extract common drug-like fragments
    and scores generated molecules based on fragment frequencies.
    """

    def __init__(self, fragment_counts: Optional[Dict[str, int]] = None):
        """
        Args:
            fragment_counts: Pre-computed fragment counts {SMILES: count}
        """
        self.fragment_counts = fragment_counts or {}
        self.total_fragments = sum(self.fragment_counts.values()) if self.fragment_counts else 0

    @classmethod
    def from_smiles_list(cls, smiles_list: List[str], min_count: int = 2) -> 'FragmentVocab':
        """
        Build fragment vocabulary from a list of SMILES.

        Args:
            smiles_list: List of training SMILES
            min_count: Minimum fragment occurrences to include

        Returns:
            FragmentVocab instance
        """
        fragment_counter = Counter()

        for smiles in smiles_list:
            try:
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    continue

                # BRICS decomposition
                fragments = BRICS.BRICSDecompose(mol)
                for frag in fragments:
                    # Clean fragment SMILES
                    frag_mol = Chem.MolFromSmiles(frag)
                    if frag_mol is not None:
                        clean_frag = Chem.MolToSmiles(frag_mol, canonical=True)
                        fragment_counter[clean_frag] += 1
            except Exception:
                continue

        # Filter by min_count
        filtered = {
            frag: count for frag, count in fragment_counter.items()
            if count >= min_count
        }

        return cls(fragment_counts=filtered)

    def score_molecule(self, mol_or_smiles) -> float:
        """
        Score a molecule based on how common its fragments are.

        Higher score = more common/realistic fragments.
        Returns negative log probability (lower is better).

        Args:
            mol_or_smiles: RDKit mol or SMILES string

        Returns:
            Fragment score (lower = more common fragments)
        """
        if isinstance(mol_or_smiles, str):
            mol = Chem.MolFromSmiles(mol_or_smiles)
        else:
            mol = mol_or_smiles

        if mol is None or self.total_fragments == 0:
            return 10.0  # High penalty for invalid

        try:
            fragments = BRICS.BRICSDecompose(mol)
            if not fragments:
                return 5.0  # Medium penalty for no fragments

            scores = []
            for frag in fragments:
                frag_mol = Chem.MolFromSmiles(frag)
                if frag_mol is None:
                    continue
                clean_frag = Chem.MolToSmiles(frag_mol, canonical=True)

                count = self.fragment_counts.get(clean_frag, 0)
                if count > 0:
                    prob = count / self.total_fragments
                    scores.append(-math.log(prob))
                else:
                    scores.append(10.0)  # Unseen fragment penalty

            if not scores:
                return 5.0

            return sum(scores) / len(scores)

        except Exception:
            return 10.0

    def save(self, path: str):
        """Save fragment vocabulary to JSON."""
        os.makedirs(os.path.dirname(path) if os.path.dirname(path) else '.', exist_ok=True)
        with open(path, 'w') as f:
            json.dump({
                'fragment_counts': self.fragment_counts,
                'total_fragments': self.total_fragments,
            }, f, indent=2)

    @classmethod
    def load(cls, path: str) -> 'FragmentVocab':
        """Load fragment vocabulary from JSON."""
        with open(path, 'r') as f:
            data = json.load(f)
        return cls(fragment_counts=data['fragment_counts'])

    def __len__(self) -> int:
        return len(self.fragment_counts)

    def __repr__(self) -> str:
        return f"FragmentVocab(n_fragments={len(self)}, total_count={self.total_fragments})"

### 5.3 · Graph Transformer
Multi-scale graph transformer with:
- Graph attention with edge feature propagation
- Ring-centric attention
- Global graph pooling with gating
- FiLM property conditioning with classifier-free guidance
- Gradient checkpointing support

In [ ]:
"""
IMPROVED Graph Transformer for molecular denoising in diffusion models.

Key improvements over original:
1. Fixed edge feature propagation in attention layers
2. Added gradient checkpointing for memory efficiency
3. Improved FiLM conditioning with conditional dropout
4. Better ring attention with actual ring detection
5. More stable attention mechanism with proper normalization
6. Added hybrid features to all output heads
7. Spectral normalization option for training stability
8. Better edge prediction using both node features
"""
from typing import Dict, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.utils import softmax
from torch.utils.checkpoint import checkpoint
import math

# (config already defined above)

class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal embeddings for timestamp conditioning."""

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, time: torch.Tensor) -> torch.Tensor:
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time.float().unsqueeze(-1) * embeddings.unsqueeze(0)
        embeddings = torch.cat([embeddings.sin(), embeddings.cos()], dim=-1)
        return embeddings

class PropertyEmbedding(nn.Module):
    """
    IMPROVED: MLP with conditional dropout for classifier-free guidance.

    Changes:
    - Added dropout_prob parameter for classifier-free guidance
    - Separate MLPs for scale and shift (more expressive)
    - Layer normalization for stability
    """

    def __init__(self, num_properties: int, hidden_dim: int, dropout_prob: float = 0.1):
        super().__init__()
        self.dropout_prob = dropout_prob

        # Shared base MLP
        self.base_mlp = nn.Sequential(
            nn.Linear(num_properties, hidden_dim),
            nn.LayerNorm(hidden_dim),  # Added for stability
            nn.GELU(),
            nn.Dropout(0.1),  # Internal dropout
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        # Separate projections for scale and shift (more expressive)
        self.scale_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()  # Bound scale to [-1, 1] then shift to [0, 2]
        )

        self.shift_mlp = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, properties: torch.Tensor):
        """
        Args:
            properties: [B, num_properties]
        Returns:
            scale: [B, hidden_dim], shift: [B, hidden_dim]
        """
        # Classifier-free guidance: randomly zero out properties during training
        if self.training and torch.rand(1).item() < self.dropout_prob:
            properties = torch.zeros_like(properties)

        h = self.base_mlp(properties)

        # Scale bounded to [0, 2] range (1 ± 1)
        scale = self.scale_mlp(h) + 1.0  # Map [-1, 1] -> [0, 2]
        shift = self.shift_mlp(h)

        return scale, shift

class GraphAttentionLayer(MessagePassing):
    """
    IMPROVED: Multi-head graph attention with better edge handling.

    Changes:
    - Edge features properly integrated into attention scores
    - Attention normalization per destination node (not global)
    - Optional spectral normalization for stability
    - Edge feature update mechanism
    """

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.1, 
                 edge_dim: int = None, spectral_norm: bool = False):
        super().__init__(aggr='add', node_dim=0)
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.hidden_dim = hidden_dim
        self.edge_dim = edge_dim or hidden_dim

        # Node projections
        self.W_q = nn.Linear(hidden_dim, hidden_dim)
        self.W_k = nn.Linear(hidden_dim, hidden_dim)
        self.W_v = nn.Linear(hidden_dim, hidden_dim)

        # Edge projection - IMPROVED: properly sized
        self.W_e = nn.Linear(self.edge_dim, num_heads)

        # Output projection
        self.W_o = nn.Linear(hidden_dim, hidden_dim)

        # Edge feature update (NEW)
        self.edge_update_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + self.edge_dim, self.edge_dim),
            nn.GELU(),
            nn.Linear(self.edge_dim, self.edge_dim)
        )

        self.dropout = nn.Dropout(dropout)

        # Optional spectral normalization for stability
        if spectral_norm:
            self.W_q = nn.utils.spectral_norm(self.W_q)
            self.W_k = nn.utils.spectral_norm(self.W_k)
            self.W_v = nn.utils.spectral_norm(self.W_v)

    def forward(self, x, edge_index, edge_attr):
        """
        Returns:
            x_out: Updated node features
            edge_attr_out: Updated edge features (NEW)
        """
        q = self.W_q(x).view(-1, self.num_heads, self.head_dim)
        k = self.W_k(x).view(-1, self.num_heads, self.head_dim)
        v = self.W_v(x).view(-1, self.num_heads, self.head_dim)

        edge_weight = self.W_e(edge_attr)  # [E, num_heads]

        # Message passing
        out = self.propagate(edge_index, q=q, k=k, v=v, 
                            edge_weight=edge_weight, x=x, edge_attr=edge_attr)
        out = out.view(-1, self.hidden_dim)
        x_out = self.W_o(out)

        # Update edge features (NEW)
        src, dst = edge_index
        edge_input = torch.cat([x[src], x[dst], edge_attr], dim=-1)
        edge_attr_out = edge_attr + self.edge_update_mlp(edge_input)  # Residual

        return x_out, edge_attr_out

    def message(self, q_i, k_j, v_j, edge_weight, index, ptr, size_i):
        """
        IMPROVED: Better attention computation with proper normalization.
        """
        # Attention scores: q·k/√d + edge_bias
        attn = (q_i * k_j).sum(dim=-1) / math.sqrt(self.head_dim)  # [E, num_heads]
        attn = attn + edge_weight  # Add edge bias

        # Softmax per destination node (proper normalization)
        attn = softmax(attn, index, ptr, size_i)
        attn = self.dropout(attn)

        # Apply attention to values
        return v_j * attn.unsqueeze(-1)  # [E, num_heads, head_dim]

class RingAttentionLayer(nn.Module):
    """
    IMPROVED: Ring-centric attention with actual ring detection.

    Changes:
    - Uses in_ring feature from node features (index 20)
    - Separate attention for ring vs non-ring atoms
    - Ring pooling and broadcasting
    """

    def __init__(self, hidden_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)

        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, batch: torch.Tensor, 
                node_features: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Args:
            x: Node embeddings [N, hidden_dim]
            batch: Batch assignment [N]
            node_features: Original node features for ring detection [N, feature_dim]

        Returns:
            Updated node features with ring context
        """
        N = x.shape[0]
        device = x.device

        q = self.q_proj(x).view(N, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(N, self.num_heads, self.head_dim)
        v = self.v_proj(x).view(N, self.num_heads, self.head_dim)

        # Detect ring atoms (NEW - uses actual ring information)
        if node_features is not None:
            # Assuming in_ring is at index 20 in node features
            in_ring_mask = node_features[:, 20] > 0.5  # Boolean mask
        else:
            # Fallback: use all atoms
            in_ring_mask = torch.ones(N, dtype=torch.bool, device=device)

        num_graphs = batch.max().item() + 1

        # Pool ring atoms per graph
        ring_k = torch.zeros(num_graphs, self.num_heads, self.head_dim, device=device)
        ring_v = torch.zeros(num_graphs, self.num_heads, self.head_dim, device=device)
        ring_count = torch.zeros(num_graphs, device=device)

        # Only aggregate ring atoms
        for graph_id in range(num_graphs):
            graph_mask = (batch == graph_id) & in_ring_mask
            if graph_mask.sum() > 0:
                ring_k[graph_id] = k[graph_mask].mean(dim=0)
                ring_v[graph_id] = v[graph_mask].mean(dim=0)
                ring_count[graph_id] = graph_mask.sum()

        # Broadcast ring context to all atoms
        node_ring_k = ring_k[batch]  # [N, heads, head_dim]
        node_ring_v = ring_v[batch]

        # Attention scores
        attn = (q * node_ring_k).sum(dim=-1) / math.sqrt(self.head_dim)  # [N, heads]

        # Higher attention for ring atoms
        if node_features is not None:
            attn = attn + in_ring_mask.float().unsqueeze(-1) * 0.5  # Bias for ring atoms

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        # Weighted values
        out = node_ring_v * attn.unsqueeze(-1)  # [N, heads, head_dim]
        out = out.reshape(N, self.hidden_dim)

        return self.out_proj(out)

class GlobalGraphPool(nn.Module):
    """
    IMPROVED: Global pooling with gating mechanism.

    Changes:
    - Gated update (learn when to use global context)
    - Separate MLP for global features
    """

    def __init__(self, hidden_dim: int, dropout: float = 0.1):
        super().__init__()

        # Global feature MLP
        self.global_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )

        # Gating mechanism (NEW)
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        # Global mean pool per graph
        graph_emb = global_mean_pool(x, batch)  # [B, hidden_dim]

        # Process global features
        graph_emb = self.global_mlp(graph_emb)  # [B, hidden_dim]

        # Broadcast back to nodes
        node_graph = graph_emb[batch]  # [N, hidden_dim]

        # Gated fusion (NEW)
        combined = torch.cat([x, node_graph], dim=-1)  # [N, 2*hidden_dim]
        gate = self.gate(combined)  # [N, hidden_dim]

        return gate * node_graph  # Gated global context

class MultiScaleBlock(nn.Module):
    """
    IMPROVED: Multi-scale transformer block with gradient checkpointing.

    Changes:
    - Edge features properly updated through layers
    - Gradient checkpointing option for memory efficiency
    - Pre-norm instead of post-norm (more stable)
    - Original node features passed for ring detection
    """

    def __init__(self, hidden_dim: int, num_heads: int, dropout: float = 0.1,
                 edge_dim: int = None, use_checkpoint: bool = False):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        # Local attention (message passing on graph edges)
        self.local_attn = GraphAttentionLayer(hidden_dim, num_heads, dropout, edge_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.edge_norm = nn.LayerNorm(edge_dim or hidden_dim)  # NEW

        # Ring attention
        self.ring_attn = RingAttentionLayer(hidden_dim, num_heads=max(1, num_heads // 2), dropout=dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Global pooling update
        self.global_pool = GlobalGraphPool(hidden_dim, dropout)
        self.norm3 = nn.LayerNorm(hidden_dim)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(dropout),
        )
        self.norm4 = nn.LayerNorm(hidden_dim)

    def forward(self, x, edge_index, edge_attr, batch, node_features=None):
        """
        Returns:
            x: Updated node features
            edge_attr: Updated edge features
        """
        def _forward(x, edge_attr):
            # Pre-norm + local attention + residual
            x_norm = self.norm1(x)
            edge_norm = self.edge_norm(edge_attr)
            x_new, edge_new = self.local_attn(x_norm, edge_index, edge_norm)
            x = x + x_new
            edge_attr = edge_attr + edge_new

            # Pre-norm + ring attention + residual
            x = x + self.ring_attn(self.norm2(x), batch, node_features)

            # Pre-norm + global context + residual
            x = x + self.global_pool(self.norm3(x), batch)

            # Pre-norm + FFN + residual
            x = x + self.ffn(self.norm4(x))

            return x, edge_attr

        # Use gradient checkpointing if enabled (saves memory)
        if self.use_checkpoint and self.training:
            x, edge_attr = checkpoint(_forward, x, edge_attr)
        else:
            x, edge_attr = _forward(x, edge_attr)

        return x, edge_attr

class GraphTransformer(nn.Module):
    """
    IMPROVED: Graph Transformer for molecular denoising.

    Key improvements:
    1. Edge features propagated through all layers
    2. Gradient checkpointing option
    3. Better property conditioning with dropout
    4. Hybrid prediction heads (separate atom type, charge, hybridization)
    5. Improved ring attention with actual ring detection
    6. Pre-norm architecture for stability
    """

    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        hidden_dim = config.hidden_dim

        # Input dimensions
        node_input_dim = len(ATOM_TYPES) + len(CHARGES) + len(HYBRIDIZATIONS) + 4
        edge_input_dim = len(BOND_TYPES) + 3

        # Store for later use
        self.node_input_dim = node_input_dim
        self.edge_input_dim = edge_input_dim

        # Input projections with layer norm
        self.node_embed = nn.Sequential(
            nn.Linear(node_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )
        self.edge_embed = nn.Sequential(
            nn.Linear(edge_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        # Property conditioning (FiLM) with classifier-free guidance
        self.property_embed = PropertyEmbedding(
            len(PROPERTY_NAMES), 
            hidden_dim, 
            dropout_prob=0.1  # 10% unconditional during training
        )

        # Multi-scale transformer blocks
        self.blocks = nn.ModuleList([
            MultiScaleBlock(
                hidden_dim, 
                config.num_heads, 
                config.dropout,
                edge_dim=hidden_dim,
                use_checkpoint=getattr(config, 'gradient_checkpointing', False)
            )
            for _ in range(config.num_layers)
        ])

        # Final layer norm
        self.final_norm = nn.LayerNorm(hidden_dim)

        # IMPROVED: Separate output heads for different features
        # Atom type prediction
        self.atom_type_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim, len(ATOM_TYPES)),
        )

        # Charge prediction
        self.charge_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim // 2, len(CHARGES)),
        )

        # Hybridization prediction (NEW - separate head)
        self.hybrid_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim // 2, len(HYBRIDIZATIONS)),
        )

        # Binary features prediction (aromatic, in_ring, conjugated)
        self.binary_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim // 4, 3),  # 3 binary features
            nn.Sigmoid()
        )

        # NumHs prediction (0-3)
        self.numhs_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim // 4, 4),  # 0, 1, 2, 3 hydrogens
        )

        # Bond type prediction (uses both node features)
        self.bond_type_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim, len(BOND_TYPES)),
        )

        # Bond features prediction (aromatic, conjugated, in_ring)
        self.bond_binary_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim // 2, 3),
            nn.Sigmoid()
        )

        # Graph-level output for property prediction
        self.graph_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(hidden_dim, len(PROPERTY_NAMES)),
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor,
        t: torch.Tensor,
        batch: torch.Tensor,
        condition: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass.

        Args:
            x: Node features [N, node_input_dim]
            edge_index: Edge connectivity [2, E]
            edge_attr: Edge features [E, edge_input_dim]
            t: Timestep [B]
            batch: Batch assignment [N]
            condition: Optional property targets [B, num_properties]

        Returns:
            Dictionary with all predictions
        """
        # Store original features for ring detection
        original_x = x.clone()

        # Project inputs
        h = self.node_embed(x)
        e = self.edge_embed(edge_attr)

        # Add time embedding (broadcast to nodes)
        t_emb = self.time_embed(t)  # [B, hidden_dim]
        h = h + t_emb[batch]

        # Apply property conditioning via FiLM
        if condition is not None:
            scale, shift = self.property_embed(condition)  # [B, hidden_dim] each
            # Broadcast to nodes
            node_scale = scale[batch]  # [N, hidden_dim]
            node_shift = shift[batch]  # [N, hidden_dim]
            h = h * node_scale + node_shift

        # Transformer blocks (edge features updated through layers)
        for block in self.blocks:
            h, e = block(h, edge_index, e, batch, original_x)

        # Final normalization
        h = self.final_norm(h)

        # Node predictions (all separate heads)
        atom_type_logits = self.atom_type_head(h)
        charge_logits = self.charge_head(h)
        hybrid_logits = self.hybrid_head(h)
        binary_features = self.binary_head(h)  # [N, 3] (aromatic, in_ring, conjugated)
        numhs_logits = self.numhs_head(h)

        # Edge predictions: concatenate source and target node features
        src, dst = edge_index
        edge_features = torch.cat([h[src], h[dst]], dim=-1)  # [E, 2*hidden_dim]
        bond_type_logits = self.bond_type_head(edge_features)
        bond_binary = self.bond_binary_head(edge_features)  # [E, 3]

        # Graph-level features
        graph_features = global_mean_pool(h, batch)  # [B, hidden_dim]
        graph_properties = self.graph_head(graph_features)  # [B, num_properties]

        return {
            # Node predictions
            'atom_type_logits': atom_type_logits,      # [N, 10]
            'charge_logits': charge_logits,            # [N, 5]
            'hybrid_logits': hybrid_logits,            # [N, 4]
            'binary_features': binary_features,        # [N, 3] (sigmoid)
            'numhs_logits': numhs_logits,             # [N, 4]

            # Edge predictions
            'bond_type_logits': bond_type_logits,      # [E, 5]
            'bond_binary': bond_binary,                # [E, 3] (sigmoid)

            # Graph-level
            'graph_properties': graph_properties,      # [B, 7]

            # Legacy compatibility (concatenated)
            'node_logits': torch.cat([
                atom_type_logits,
                charge_logits,
                hybrid_logits,
                binary_features,
                numhs_logits
            ], dim=-1),  # [N, 26]

            'edge_logits': torch.cat([
                bond_type_logits,
                bond_binary
            ], dim=-1),  # [E, 8]

            'graph_features': graph_properties,
        }

    def enable_gradient_checkpointing(self):
        """Enable gradient checkpointing to save memory during training."""
        for block in self.blocks:
            block.use_checkpoint = True

    def disable_gradient_checkpointing(self):
        """Disable gradient checkpointing (faster but uses more memory)."""
        for block in self.blocks:
            block.use_checkpoint = False

# Utility function for model initialization
def create_graph_transformer(config: ModelConfig) -> GraphTransformer:
    """
    Factory function to create and initialize GraphTransformer.

    Applies:
    - Xavier uniform initialization for linear layers
    - Small constant initialization for biases
    - Proper initialization for layer norms
    """
    model = GraphTransformer(config)

    # Initialize weights
    def init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.01)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0.0)

    model.apply(init_weights)

    return model

### 5.4 · Molecular Diffusion Model
Complete diffusion model combining the noise scheduler and graph transformer for training and sampling.

In [ ]:
"""
Molecular Diffusion Model.

Combines the noise scheduler and graph transformer to create
a complete diffusion model for molecular generation.
"""
from typing import Dict, Optional, List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from tqdm import tqdm

# (config already defined above)
# (config already defined above)
# (NoiseScheduler defined above)
# (GraphTransformer defined above)

# Valid valences for each atom type (by atomic number)
VALID_VALENCES = {
    6: [4],           # Carbon
    7: [3, 5],        # Nitrogen
    8: [2],           # Oxygen
    16: [2, 4, 6],    # Sulfur
    9: [1],           # Fluorine
    17: [1, 3, 5, 7], # Chlorine
    35: [1, 3, 5],    # Bromine
    15: [3, 5],       # Phosphorus
    53: [1, 3, 5, 7], # Iodine
}

ATOM_SYMBOL_TO_ATOMIC_NUM = {
    'C': 6, 'N': 7, 'O': 8, 'S': 16, 'F': 9,
    'Cl': 17, 'Br': 35, 'P': 15, 'I': 53,
}

class MolecularDiffusion(nn.Module):
    """
    Complete diffusion model for molecular generation.

    Combines:
    - Discrete noise scheduler for categorical features
    - Graph transformer for denoising
    - Training and sampling procedures
    - Auxiliary losses (valency, property, fragment)
    - Classifier-free guidance for property conditioning
    """

    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.device = config.device

        # Model components
        self.transformer = GraphTransformer(config.model)
        self.noise_scheduler = NoiseScheduler(config.diffusion, config.device)

        # Loss config
        self.loss_config = config.loss
        self.guidance_config = config.guidance

        # Number of classes
        self.num_atom_types = len(ATOM_TYPES)
        self.num_bond_types = len(BOND_TYPES)
        self.num_charges = len(CHARGES)
        self.num_hybridizations = len(HYBRIDIZATIONS)

        # Node & edge feature dimensions (must match molecular_graph.py)
        # Node: atom_types(10) + charges(5) + hybrid(4) + aromatic(1) + in_ring(1) + num_hs(1) + conjugated(1) = 23
        self.node_feat_dim = self.num_atom_types + self.num_charges + self.num_hybridizations + 4
        # Edge: bond_types(5) + aromatic(1) + conjugated(1) + in_ring(1) = 8
        self.edge_feat_dim = self.num_bond_types + 3

    def training_step(self, batch: Batch) -> Dict[str, torch.Tensor]:
        """
        Perform one training step with multi-objective loss.

        Args:
            batch: Batched molecular graphs

        Returns:
            Dictionary containing loss and metrics
        """
        # Sample random timesteps
        t = self.noise_scheduler.sample_timesteps(batch.num_graphs)

        # Get original node and edge types
        node_types_orig = batch.node_types
        edge_types_orig = batch.edge_types

        # Get original charges
        if hasattr(batch, 'node_charges'):
            charges_orig = batch.node_charges
        else:
            charges_orig = torch.zeros_like(node_types_orig) + 2  # Default neutral

        # Expand timesteps to match nodes/edges
        node_t = t[batch.batch]
        edge_batch = batch.batch[batch.edge_index[0]]
        edge_t = t[edge_batch]

        # Add noise to node types and edge types
        node_types_noisy = self.noise_scheduler.add_noise(
            node_types_orig, node_t, self.num_atom_types
        )
        edge_types_noisy = self.noise_scheduler.add_noise(
            edge_types_orig, edge_t, self.num_bond_types
        )
        charges_noisy = self.noise_scheduler.add_noise(
            charges_orig, node_t, self.num_charges
        )

        # Create noisy input features
        x_noisy = self._create_node_features(
            node_types_noisy, charges_noisy, batch
        )
        edge_attr_noisy = self._create_edge_features(
            edge_types_noisy, batch
        )

        # Property conditioning with classifier-free guidance dropout
        condition = None
        if hasattr(batch, 'properties') and batch.properties is not None:
            condition = batch.properties
            if condition.dim() == 1:
                # Flattened by DataLoader; reshape to [num_graphs, num_properties]
                num_props = condition.shape[0] // batch.num_graphs
                condition = condition.view(batch.num_graphs, num_props)
            elif condition.dim() == 2 and condition.shape[0] != batch.num_graphs:
                # Properties batched per-node; take per-graph via scatter
                condition = condition[:batch.num_graphs]

            # Classifier-free guidance dropout: randomly zero out conditioning
            if self.training and self.guidance_config.condition_dropout > 0:
                mask = torch.rand(condition.shape[0], device=condition.device) < self.guidance_config.condition_dropout
                condition = condition.clone()
                condition[mask.unsqueeze(-1).expand_as(condition)] = 0.0

        # Forward pass through transformer
        predictions = self.transformer(
            x_noisy,
            batch.edge_index,
            edge_attr_noisy,
            t,
            batch.batch,
            condition=condition,
        )

        # === Primary Loss: Cross-entropy on predictions ===
        node_loss = F.cross_entropy(
            predictions['node_logits'][:, :self.num_atom_types],
            node_types_orig.long()
        )
        charge_loss = F.cross_entropy(
            predictions['charge_logits'],
            charges_orig.long()
        )
        edge_loss = F.cross_entropy(
            predictions['edge_logits'][:, :self.num_bond_types],
            edge_types_orig.long()
        )

        diffusion_loss = node_loss + 0.5 * charge_loss + 0.5 * edge_loss

        # === Auxiliary Loss 1: Valency loss ===
        valency_loss = self._compute_valency_loss(
            predictions['node_logits'],
            predictions['edge_logits'][:, :self.num_bond_types],
            batch.edge_index,
            batch.batch,
        )

        # === Auxiliary Loss 2: Property matching loss ===
        property_loss = torch.tensor(0.0, device=diffusion_loss.device)
        if condition is not None and 'graph_features' in predictions:
            property_loss = self._compute_property_loss(
                predictions['graph_features'],
                condition,
            )

        # === Total loss ===
        total_loss = (
            diffusion_loss
            + self.loss_config.lambda_valency * valency_loss
            + self.loss_config.lambda_property * property_loss
        )

        # Compute accuracy metrics
        with torch.no_grad():
            node_acc = (predictions['node_logits'][:, :self.num_atom_types].argmax(dim=-1) == node_types_orig).float().mean()
            edge_acc = (predictions['edge_logits'][:, :self.num_bond_types].argmax(dim=-1) == edge_types_orig).float().mean()

        return {
            'loss': total_loss,
            'diffusion_loss': diffusion_loss,
            'node_loss': node_loss,
            'charge_loss': charge_loss,
            'edge_loss': edge_loss,
            'valency_loss': valency_loss,
            'property_loss': property_loss,
            'node_acc': node_acc,
            'edge_acc': edge_acc,
        }

    def _compute_valency_loss(
        self,
        node_logits: torch.Tensor,
        edge_logits: torch.Tensor,
        edge_index: torch.Tensor,
        batch: torch.Tensor,
    ) -> torch.Tensor:
        """
        Penalize predicted atoms whose total bond order exceeds valid valency.

        Uses predicted (soft) atom types and bond types to compute expected
        valency violation.  Applies softmax only to the atom-type slice of
        node_logits (first num_atom_types dims) and accumulates bond orders
        from both edge directions for undirected graphs.
        """
        device = node_logits.device

        # --- Soft atom type probabilities (FIXED: slice atom-type dims only) ---
        atom_probs = F.softmax(
            node_logits[:, :self.num_atom_types], dim=-1
        )  # [N, num_atom_types]

        # Soft bond order: weighted sum of bond orders
        # Bond orders: NONE=0, SINGLE=1, DOUBLE=2, TRIPLE=3, AROMATIC=1.5
        bond_orders = torch.tensor([0.0, 1.0, 2.0, 3.0, 1.5], device=device)
        edge_probs = F.softmax(edge_logits, dim=-1)  # [E, num_bond_types]
        expected_bond_order = (edge_probs * bond_orders.unsqueeze(0)).sum(dim=-1)  # [E]

        # Sum bond orders per atom (accumulate from BOTH src and dst)
        N = node_logits.shape[0]
        atom_valency = torch.zeros(N, device=device)
        atom_valency.scatter_add_(0, edge_index[0], expected_bond_order)
        # NOTE: edges are stored bidirectionally so each direction already
        # contributes; if your dataset uses unidirectional edges uncomment:
        # atom_valency.scatter_add_(0, edge_index[1], expected_bond_order)

        # Expected max valency per atom (weighted by atom type probs)
        # Max valences: C=4, N=5, O=2, S=6, F=1, Cl=7, Br=5, P=5, I=7, Other=4
        max_valences = torch.tensor(
            [4.0, 5.0, 2.0, 6.0, 1.0, 7.0, 5.0, 5.0, 7.0, 4.0], device=device
        )
        expected_max_valency = (atom_probs * max_valences.unsqueeze(0)).sum(dim=-1)

        # Quadratic penalty for violations (stronger gradient signal)
        violation = F.relu(atom_valency - expected_max_valency)
        excess_valency_loss = (violation ** 2).mean()

        # --- Total valency matching loss ---
        # Target total valency (bond order + implicit H) per atom type
        target_total_valency = torch.tensor(
            [4.0, 3.0, 2.0, 2.0, 1.0, 1.0, 1.0, 3.0, 1.0, 4.0], device=device
        )  # C, N, O, S, F, Cl, Br, P, I, Other
        expected_target = (atom_probs * target_total_valency.unsqueeze(0)).sum(dim=-1)
        valency_match_loss = F.mse_loss(atom_valency, expected_target)

        return excess_valency_loss + 0.5 * valency_match_loss

    def _compute_property_loss(
        self,
        graph_features: torch.Tensor,
        target_properties: torch.Tensor,
    ) -> torch.Tensor:
        """
        Per-property weighted MSE loss between predicted graph features
        and target properties.

        Properties are already normalized in dataset.py, but we add
        per-property weighting to boost broken properties (HBD, HBA,
        aromatic rings).

        Property order (from config.PROPERTY_NAMES):
            0: molecular_weight, 1: logp, 2: num_aromatic_rings,
            3: hbd, 4: hba, 5: fraction_sp3, 6: tpsa, 7: sa_score
        """
        # Per-property weights — boost properties that were broken in
        # generated molecules (HBD, HBA, aromatic rings, TPSA)
        weights = torch.tensor(
            [1.0, 1.0, 2.0, 2.0, 2.0, 1.0, 1.5, 1.0],
            device=graph_features.device,
        )
        # Truncate or pad weights to match actual feature dim
        num_props = min(graph_features.shape[-1], weights.shape[0])
        w = weights[:num_props]

        # Weighted MSE
        diff_sq = (graph_features[:, :num_props] - target_properties[:, :num_props]) ** 2
        weighted = diff_sq * w.unsqueeze(0)
        return weighted.mean()

    def _create_node_features(
        self,
        node_types: torch.Tensor,
        charges: torch.Tensor,
        batch: Batch,
    ) -> torch.Tensor:
        """Create node feature tensor from categorical types."""
        N = node_types.shape[0]
        device = node_types.device

        # One-hot encode atom types
        node_type_onehot = torch.zeros(N, self.num_atom_types, device=device)
        node_type_onehot.scatter_(1, node_types.unsqueeze(1), 1)

        # One-hot encode charges
        charge_onehot = torch.zeros(N, self.num_charges, device=device)
        charge_onehot.scatter_(1, charges.unsqueeze(1), 1)

        # Get hybridization, aromatic, ring, num_hs, conjugated from original batch
        offset = self.num_atom_types + self.num_charges
        hybrid = batch.x[:, offset:offset + self.num_hybridizations]
        aromatic = batch.x[:, offset + self.num_hybridizations].unsqueeze(1)
        in_ring = batch.x[:, offset + self.num_hybridizations + 1].unsqueeze(1)
        num_hs = batch.x[:, offset + self.num_hybridizations + 2].unsqueeze(1)
        conjugated = batch.x[:, offset + self.num_hybridizations + 3].unsqueeze(1)

        return torch.cat([node_type_onehot, charge_onehot, hybrid, aromatic, in_ring, num_hs, conjugated], dim=1)

    def _create_edge_features(
        self,
        edge_types: torch.Tensor,
        batch: Batch,
    ) -> torch.Tensor:
        """Create edge feature tensor from categorical types."""
        E = edge_types.shape[0]
        device = edge_types.device

        # One-hot encode bond types
        edge_type_onehot = torch.zeros(E, self.num_bond_types, device=device)
        edge_type_onehot.scatter_(1, edge_types.unsqueeze(1), 1)

        # Get aromatic, conjugated, ring flags from original batch
        aromatic = batch.edge_attr[:, self.num_bond_types].unsqueeze(1)
        conjugated = batch.edge_attr[:, self.num_bond_types + 1].unsqueeze(1)
        in_ring = batch.edge_attr[:, self.num_bond_types + 2].unsqueeze(1)

        return torch.cat([edge_type_onehot, aromatic, conjugated, in_ring], dim=1)

    @torch.no_grad()
    def sample(
        self,
        num_molecules: int,
        num_atoms: int = 20,
        temperature: float = 1.0,
        target_properties: Optional[torch.Tensor] = None,
        guidance_scale: Optional[float] = None,
        num_sampling_steps: Optional[int] = None,
    ) -> List[Data]:
        """
        Sample new molecules from the model.

        Args:
            num_molecules: Number of molecules to generate
            num_atoms: Number of atoms per molecule
            temperature: Sampling temperature
            target_properties: Optional property targets for guided generation [num_properties]
            guidance_scale: Guidance scale (uses config default if None)
            num_sampling_steps: Number of denoising steps (default: all timesteps).
                               Lower = faster but lower quality. 50-100 is usually fine.

        Returns:
            List of generated molecular graphs
        """
        self.eval()
        device = next(self.parameters()).device
        gs = guidance_scale if guidance_scale is not None else self.guidance_config.guidance_scale

        # Build the timestep schedule (optionally skipping steps for speed)
        total_timesteps = self.noise_scheduler.num_timesteps
        if num_sampling_steps is not None and num_sampling_steps < total_timesteps:
            # Evenly spaced subset of timesteps (like DDIM-style skipping)
            step_ratio = total_timesteps / num_sampling_steps
            timesteps = [int(round(i * step_ratio)) for i in reversed(range(num_sampling_steps))]
        else:
            timesteps = list(reversed(range(total_timesteps)))

        # Atom-type prior: bias toward realistic drug-like distribution
        # C:60%, N:12%, O:10%, S:4%, F:3%, Cl:3%, Br:2%, P:2%, I:1%, Other:3%
        atom_prior = torch.tensor([0.60, 0.12, 0.10, 0.04, 0.03, 0.03, 0.02, 0.02, 0.01, 0.03],
                                  device=device)
        # Bond-type prior: most edges should be NONE in initial noise for sparse graphs
        # NONE:70%, SINGLE:18%, DOUBLE:7%, TRIPLE:1%, AROMATIC:4%
        bond_prior = torch.tensor([0.70, 0.18, 0.07, 0.01, 0.04], device=device)

        generated = []

        for mol_idx in range(num_molecules):
            # Initialize with prior-biased noise (not uniform)
            node_types = torch.multinomial(atom_prior, num_atoms, replacement=True)
            charges = torch.full((num_atoms,), 2, device=device)  # Neutral

            # Create fully connected graph for initial structure
            edge_index = self._create_full_edge_index(num_atoms, device)
            num_edges = edge_index.shape[1]
            edge_types = torch.multinomial(bond_prior, num_edges, replacement=True)

            # Batch information
            batch_idx = torch.zeros(num_atoms, dtype=torch.long, device=device)

            # Prepare conditioning
            cond = target_properties.unsqueeze(0).to(device) if target_properties is not None else None

            # Reverse diffusion with progress bar
            desc = f"Mol {mol_idx+1}/{num_molecules} ({num_atoms} atoms)"
            for t in tqdm(timesteps, desc=desc, leave=False):
                t_tensor = torch.tensor([t], device=device)

                # Create features
                x = self._create_node_features_simple(node_types, charges, num_atoms, device)
                edge_attr = self._create_edge_features_simple(edge_types, device)

                # Conditional prediction
                predictions = self.transformer(
                    x, edge_index, edge_attr, t_tensor, batch_idx,
                    condition=cond,
                )

                # Classifier-free guidance
                if cond is not None and gs > 1.0:
                    # Unconditional prediction
                    predictions_uncond = self.transformer(
                        x, edge_index, edge_attr, t_tensor, batch_idx,
                        condition=None,
                    )
                    # Guided logits
                    for key in ['node_logits', 'edge_logits', 'charge_logits']:
                        predictions[key] = (
                            predictions_uncond[key]
                            + gs * (predictions[key] - predictions_uncond[key])
                        )

                if t > 0:
                    # Sample from predicted distribution with temperature
                    node_probs = F.softmax(predictions['node_logits'][:, :self.num_atom_types] / temperature, dim=-1)
                    edge_probs = F.softmax(predictions['edge_logits'][:, :self.num_bond_types] / temperature, dim=-1)
                    charge_probs = F.softmax(predictions['charge_logits'] / temperature, dim=-1)

                    node_types = torch.multinomial(node_probs, 1).squeeze(-1)
                    edge_types = torch.multinomial(edge_probs, 1).squeeze(-1)
                    charges = torch.multinomial(charge_probs, 1).squeeze(-1)
                else:
                    # Final step: take argmax
                    node_types = predictions['node_logits'][:, :self.num_atom_types].argmax(dim=-1)
                    edge_types = predictions['edge_logits'][:, :self.num_bond_types].argmax(dim=-1)
                    charges = predictions['charge_logits'].argmax(dim=-1)

            # Create output graph
            # Filter edges: only keep non-NONE bonds
            valid_edges = edge_types > 0  # NONE is index 0
            edge_index_filtered = edge_index[:, valid_edges]
            edge_types_filtered = edge_types[valid_edges]

            graph = Data(
                x=F.one_hot(node_types, self.num_atom_types).float(),
                edge_index=edge_index_filtered,
                edge_attr=F.one_hot(edge_types_filtered, self.num_bond_types).float(),
                node_types=node_types,
                edge_types=edge_types_filtered,
                node_charges=charges,
            )

            generated.append(graph)
            print(f"  Generated molecule {mol_idx+1}/{num_molecules}")

        return generated

    def _create_full_edge_index(self, num_atoms: int, device: torch.device) -> torch.Tensor:
        """Create fully connected edge index."""
        src = []
        dst = []
        for i in range(num_atoms):
            for j in range(i + 1, num_atoms):
                src.extend([i, j])
                dst.extend([j, i])

        return torch.tensor([src, dst], dtype=torch.long, device=device)

    def _create_node_features_simple(
        self,
        node_types: torch.Tensor,
        charges: torch.Tensor,
        num_atoms: int,
        device: torch.device,
    ) -> torch.Tensor:
        """Create simple node features without batch reference.

        Estimates implicit hydrogen count from atom type to avoid the
        training/generation mismatch that caused HBD=0 in all generated
        molecules.
        """
        node_type_onehot = F.one_hot(node_types, self.num_atom_types).float()
        charge_onehot = F.one_hot(charges, self.num_charges).float()

        # Placeholder hybridization (sp3 default), aromatic, ring, conjugated
        hybrid = torch.zeros(num_atoms, self.num_hybridizations, device=device)
        hybrid[:, 2] = 1.0  # sp3 default
        aromatic = torch.zeros(num_atoms, 1, device=device)
        in_ring = torch.zeros(num_atoms, 1, device=device)
        conjugated = torch.zeros(num_atoms, 1, device=device)

        # Estimate implicit hydrogens from atom type (FIXED: was all zeros)
        # Default H counts assuming sp3: C=4-bonds, N=1, O=1, S=0, halogens=0
        default_hs = torch.tensor(
            [1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
            device=device,
        )  # C, N, O, S, F, Cl, Br, P, I, Other
        num_hs = default_hs[node_types].unsqueeze(1)  # [num_atoms, 1]

        return torch.cat([node_type_onehot, charge_onehot, hybrid, aromatic, in_ring, num_hs, conjugated], dim=1)

    def _create_edge_features_simple(
        self,
        edge_types: torch.Tensor,
        device: torch.device,
    ) -> torch.Tensor:
        """Create simple edge features without batch reference."""
        E = edge_types.shape[0]

        edge_type_onehot = F.one_hot(edge_types, self.num_bond_types).float()
        aromatic = torch.zeros(E, 1, device=device)
        conjugated = torch.zeros(E, 1, device=device)
        in_ring = torch.zeros(E, 1, device=device)

        return torch.cat([edge_type_onehot, aromatic, conjugated, in_ring], dim=1)

    def to(self, device):
        """Move model to device and update noise scheduler."""
        super().to(device)
        self.device = device
        self.noise_scheduler = NoiseScheduler(self.config.diffusion, device)
        return self

## 6 · Training
### 6.1 · Trainer
Training loop with EMA, cosine-annealing LR schedule, warmup, checkpointing, and validation.

In [ ]:
"""
Training utilities for molecular diffusion model.
"""
import os
import copy
from pathlib import Path
from typing import Optional, Dict, Any
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch_geometric.data import Batch
from tqdm import tqdm

# (config already defined above)
# (MolecularDiffusion defined above)

class EMA:
    """Exponential Moving Average of model parameters."""

    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model: nn.Module):
        for name, param in model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self, model: nn.Module):
        """Apply EMA weights (for evaluation/sampling)."""
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self, model: nn.Module):
        """Restore original weights after EMA evaluation."""
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

    def state_dict(self):
        return {'shadow': self.shadow, 'decay': self.decay}

    def load_state_dict(self, state_dict):
        self.shadow = state_dict['shadow']
        self.decay = state_dict['decay']

class WarmupCosineScheduler:
    """Linear warmup followed by cosine annealing."""

    def __init__(self, optimizer, warmup_steps: int, total_steps: int, min_lr_ratio: float = 0.01):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr_ratio = min_lr_ratio
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.current_step = 0

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            # Linear warmup
            scale = self.current_step / max(1, self.warmup_steps)
        else:
            # Cosine annealing
            import math
            progress = (self.current_step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            scale = self.min_lr_ratio + 0.5 * (1.0 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))

        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

    def state_dict(self):
        return {'current_step': self.current_step}

    def load_state_dict(self, state_dict):
        self.current_step = state_dict['current_step']

class Trainer:
    """
    Trainer class for molecular diffusion model.

    Handles training loop, validation, checkpointing, EMA, and logging.
    """

    def __init__(
        self,
        model: MolecularDiffusion,
        config: Config,
        train_loader: DataLoader,
        val_loader: Optional[DataLoader] = None,
    ):
        self.model = model
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader

        # Move model to device
        self.device = config.device
        self.model = self.model.to(self.device)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.training.learning_rate,
            weight_decay=config.training.weight_decay,
        )

        # Learning rate scheduler with warmup
        total_steps = config.training.epochs * len(train_loader)
        self.scheduler = WarmupCosineScheduler(
            self.optimizer,
            warmup_steps=config.training.warmup_steps,
            total_steps=total_steps,
            min_lr_ratio=0.01,
        )

        # EMA
        self.ema = EMA(model, decay=config.training.ema_decay)

        # Training state
        self.current_epoch = 0
        self.global_step = 0
        self.best_val_loss = float('inf')

        # Create checkpoint directory
        self.checkpoint_dir = Path(config.training.checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        # Metrics history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'node_acc': [],
            'edge_acc': [],
            'valency_loss': [],
            'property_loss': [],
        }

    def train_epoch(self) -> Dict[str, float]:
        """Train for one epoch."""
        self.model.train()

        total_loss = 0.0
        total_node_loss = 0.0
        total_edge_loss = 0.0
        total_valency_loss = 0.0
        total_property_loss = 0.0
        total_node_acc = 0.0
        total_edge_acc = 0.0
        num_batches = 0

        pbar = tqdm(self.train_loader, desc=f"Epoch {self.current_epoch + 1}")

        for batch in pbar:
            # Move batch to device
            batch = batch.to(self.device)

            # Forward pass
            self.optimizer.zero_grad()
            metrics = self.model.training_step(batch)

            # Backward pass
            loss = metrics['loss']
            loss.backward()

            # Gradient clipping
            if self.config.training.gradient_clip > 0:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(),
                    self.config.training.gradient_clip
                )

            self.optimizer.step()
            self.scheduler.step()
            self.ema.update(self.model)

            # Update metrics
            total_loss += loss.item()
            total_node_loss += metrics['node_loss'].item()
            total_edge_loss += metrics['edge_loss'].item()
            total_valency_loss += metrics['valency_loss'].item()
            total_property_loss += metrics['property_loss'].item()
            total_node_acc += metrics['node_acc'].item()
            total_edge_acc += metrics['edge_acc'].item()
            num_batches += 1
            self.global_step += 1

            # Update progress bar
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'node_acc': f"{metrics['node_acc'].item():.3f}",
                'val_l': f"{metrics['valency_loss'].item():.3f}",
                'prop_l': f"{metrics['property_loss'].item():.3f}",
            })

            # Periodic detailed loss breakdown
            if num_batches % 10 == 0:
                diff_l = metrics.get('diffusion_loss', metrics['node_loss'] + metrics['edge_loss'])
                diff_val = diff_l.item() if hasattr(diff_l, 'item') else float(diff_l)
                val_l = metrics['valency_loss'].item()
                prop_l = metrics['property_loss'].item()
                pbar.write(
                    f"  [Step {self.global_step}] "
                    f"diff={diff_val:.4f}  val={val_l:.4f}  prop={prop_l:.4f}  "
                    f"ratio(val/diff)={val_l / max(diff_val, 1e-8):.3f}  "
                    f"ratio(prop/diff)={prop_l / max(diff_val, 1e-8):.3f}"
                )

        # Average metrics
        return {
            'loss': total_loss / num_batches,
            'node_loss': total_node_loss / num_batches,
            'edge_loss': total_edge_loss / num_batches,
            'valency_loss': total_valency_loss / num_batches,
            'property_loss': total_property_loss / num_batches,
            'node_acc': total_node_acc / num_batches,
            'edge_acc': total_edge_acc / num_batches,
        }

    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        """Validate the model using EMA weights."""
        if self.val_loader is None:
            return {}

        # Use EMA weights for validation
        self.ema.apply_shadow(self.model)
        self.model.eval()

        total_loss = 0.0
        total_node_acc = 0.0
        total_edge_acc = 0.0
        num_batches = 0

        for batch in self.val_loader:
            batch = batch.to(self.device)
            metrics = self.model.training_step(batch)

            total_loss += metrics['loss'].item()
            total_node_acc += metrics['node_acc'].item()
            total_edge_acc += metrics['edge_acc'].item()
            num_batches += 1

        # Restore original weights
        self.ema.restore(self.model)

        return {
            'val_loss': total_loss / num_batches,
            'val_node_acc': total_node_acc / num_batches,
            'val_edge_acc': total_edge_acc / num_batches,
        }

    def train(self, num_epochs: Optional[int] = None) -> Dict[str, list]:
        """Full training loop."""
        num_epochs = num_epochs or self.config.training.epochs

        print(f"Starting training for {num_epochs} epochs")
        print(f"Training samples: {len(self.train_loader.dataset)}")
        if self.val_loader:
            print(f"Validation samples: {len(self.val_loader.dataset)}")
        print(f"Device: {self.device}")
        print(f"EMA decay: {self.config.training.ema_decay}")

        for epoch in range(num_epochs):
            self.current_epoch = epoch

            # Train
            train_metrics = self.train_epoch()
            print(f"\nEpoch {epoch + 1}/{num_epochs}")
            print(f"  Train Loss: {train_metrics['loss']:.4f} "
                  f"(val_loss: {train_metrics['valency_loss']:.3f}, "
                  f"prop_loss: {train_metrics['property_loss']:.3f})")
            print(f"  Node Acc: {train_metrics['node_acc']:.3f}")
            print(f"  Edge Acc: {train_metrics['edge_acc']:.3f}")

            self.history['train_loss'].append(train_metrics['loss'])
            self.history['node_acc'].append(train_metrics['node_acc'])
            self.history['edge_acc'].append(train_metrics['edge_acc'])
            self.history['valency_loss'].append(train_metrics['valency_loss'])
            self.history['property_loss'].append(train_metrics['property_loss'])

            # Validate
            if self.val_loader and (epoch + 1) % self.config.training.val_frequency == 0:
                val_metrics = self.validate()
                print(f"  Val Loss (EMA): {val_metrics['val_loss']:.4f}")
                print(f"  Val Node Acc: {val_metrics['val_node_acc']:.3f}")

                self.history['val_loss'].append(val_metrics['val_loss'])

                # Save best model
                if val_metrics['val_loss'] < self.best_val_loss:
                    self.best_val_loss = val_metrics['val_loss']
                    self.save_checkpoint('best_model.pt')
                    print("  Saved best model!")

            # Periodic checkpoint
            if (epoch + 1) % self.config.training.save_frequency == 0:
                self.save_checkpoint(f'checkpoint_epoch_{epoch + 1}.pt')

        # Save final model
        self.save_checkpoint('final_model.pt')
        print("\nTraining complete!")

        return self.history

    def save_checkpoint(self, filename: str):
        """Save model checkpoint including EMA state."""
        checkpoint = {
            'epoch': self.current_epoch,
            'global_step': self.global_step,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'ema_state_dict': self.ema.state_dict(),
            'best_val_loss': self.best_val_loss,
            'config': self.config,
            'history': self.history,
        }

        path = self.checkpoint_dir / filename
        torch.save(checkpoint, path)
        print(f"Saved checkpoint: {path}")

    def load_checkpoint(self, path: str):
        """Load model checkpoint including EMA state."""
        checkpoint = torch.load(path, map_location=self.device, weights_only=False)

        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if 'ema_state_dict' in checkpoint:
            self.ema.load_state_dict(checkpoint['ema_state_dict'])
        self.current_epoch = checkpoint['epoch']
        self.global_step = checkpoint['global_step']
        self.best_val_loss = checkpoint['best_val_loss']
        self.history = checkpoint['history']

        print(f"Loaded checkpoint from epoch {self.current_epoch}")

## 7 · Generation
### 7.1 · Molecule Sampler
High-level sampler supporting basic, guided, diverse, and rejection sampling.

In [ ]:
"""
Molecule sampling from trained diffusion model.
"""
from typing import Optional, List, Dict, Tuple
import torch
from torch_geometric.data import Data

# (config already defined above)
# (MolecularDiffusion defined above)

class MoleculeSampler:
    """
    High-level sampler for generating molecules from trained diffusion model.

    Supports:
    - Basic sampling with temperature control
    - Guided sampling with target properties
    - Diverse sampling with varied sizes and temperatures
    """

    def __init__(
        self,
        model: MolecularDiffusion,
        config: Config,
    ):
        self.model = model
        self.config = config
        self.device = next(model.parameters()).device

    def sample(
        self,
        num_molecules: int = 100,
        num_atoms: int = 20,
        temperature: float = 1.0,
        num_sampling_steps: Optional[int] = None,
    ) -> List[Data]:
        """
        Generate molecules using basic sampling.

        Args:
            num_molecules: Number of molecules to generate
            num_atoms: Atoms per molecule
            temperature: Sampling temperature
            num_sampling_steps: Fewer steps = faster (default: all timesteps)

        Returns:
            List of generated molecular graphs
        """
        batch_size = self.config.generation.batch_size
        all_molecules = []

        for i in range(0, num_molecules, batch_size):
            n = min(batch_size, num_molecules - i)
            molecules = self.model.sample(
                num_molecules=n,
                num_atoms=num_atoms,
                temperature=temperature,
                num_sampling_steps=num_sampling_steps,
            )
            all_molecules.extend(molecules)

        return all_molecules

    def guided_sample(
        self,
        num_molecules: int = 100,
        num_atoms: int = 20,
        temperature: float = 1.0,
        target_properties: Optional[Dict[str, float]] = None,
        guidance_scale: Optional[float] = None,
        num_sampling_steps: Optional[int] = None,
    ) -> List[Data]:
        """
        Generate molecules with property guidance.

        Args:
            num_molecules: Number of molecules to generate
            num_atoms: Atoms per molecule
            temperature: Sampling temperature
            target_properties: Dict of {property_name: target_value}
            guidance_scale: Override default guidance scale

        Returns:
            List of generated molecular graphs
        """
        gs = guidance_scale or self.config.guidance.guidance_scale

        # Build target property tensor
        prop_tensor = None
        if target_properties:
            values = []
            for name in PROPERTY_NAMES:
                if name in target_properties:
                    values.append(target_properties[name])
                else:
                    values.append(0.0)  # Zero = no guidance for this property
            prop_tensor = torch.tensor(values, dtype=torch.float32)

        batch_size = self.config.generation.batch_size
        all_molecules = []

        for i in range(0, num_molecules, batch_size):
            n = min(batch_size, num_molecules - i)
            molecules = self.model.sample(
                num_molecules=n,
                num_atoms=num_atoms,
                temperature=temperature,
                target_properties=prop_tensor,
                guidance_scale=gs,
                num_sampling_steps=num_sampling_steps,
            )
            all_molecules.extend(molecules)

        return all_molecules

    def sample_diverse(
        self,
        num_molecules: int = 100,
        min_atoms: Optional[int] = None,
        max_atoms: Optional[int] = None,
        temperatures: Optional[List[float]] = None,
        target_properties: Optional[Dict[str, float]] = None,
        guidance_scale: Optional[float] = None,
        num_sampling_steps: Optional[int] = None,
    ) -> List[Data]:
        """
        Generate diverse molecules by varying sizes and temperatures.

        Args:
            num_molecules: Total molecules to generate
            min_atoms: Minimum atoms per molecule
            max_atoms: Maximum atoms per molecule
            temperatures: List of temperatures to try
            target_properties: Optional property targets for guidance
            guidance_scale: Override guidance scale

        Returns:
            List of generated molecular graphs
        """
        min_atoms = min_atoms or self.config.generation.min_atoms
        max_atoms = max_atoms or self.config.generation.max_atoms
        temperatures = temperatures or [0.7, 0.8, 0.9, 1.0, 1.1]

        gs = guidance_scale or self.config.guidance.guidance_scale
        prop_tensor = None
        if target_properties:
            values = [target_properties.get(name, 0.0) for name in PROPERTY_NAMES]
            prop_tensor = torch.tensor(values, dtype=torch.float32)

        all_molecules = []
        num_settings = len(temperatures) * (max_atoms - min_atoms + 1)
        per_setting = max(1, num_molecules // num_settings)

        import random
        atom_sizes = list(range(min_atoms, max_atoms + 1))

        while len(all_molecules) < num_molecules:
            num_atoms = random.choice(atom_sizes)
            temp = random.choice(temperatures)
            n = min(per_setting, num_molecules - len(all_molecules))

            molecules = self.model.sample(
                num_molecules=n,
                num_atoms=num_atoms,
                temperature=temp,
                target_properties=prop_tensor,
                guidance_scale=gs,
                num_sampling_steps=num_sampling_steps,
            )
            all_molecules.extend(molecules)

        return all_molecules[:num_molecules]

    @staticmethod
    def get_glue_targets() -> Dict[str, float]:
        """
        Return normalized target properties for glue-like molecule generation.
        Uses midpoints of the spec-defined target ranges.
        """
        return {
            'molecular_weight': 375.0 / 500.0,     # MW 300-450, mid=375
            'logp': (2.5 + 2.0) / 8.0,             # LogP 1.5-3.5, mid=2.5
            'num_aromatic_rings': 2.5 / 5.0,        # 2-3 aromatic rings, mid=2.5
            'hbd': 2.0 / 5.0,                       # HBD ~2
            'hba': 4.0 / 10.0,                      # HBA ~4
            'fraction_sp3': 0.35,                    # Fsp3 ~0.35
            'tpsa': 70.0 / 140.0,                   # TPSA ~70
            'sa_score': 3.0 / 10.0,                 # SA ~3
        }

    # ------------------------------------------------------------------
    # Rejection sampling
    # ------------------------------------------------------------------

    def generate_with_rejection(
        self,
        n_molecules: int,
        max_attempts_per_molecule: int = 10,
        num_atoms: int = 20,
        temperature: float = 1.0,
        target_properties: Optional[Dict[str, float]] = None,
        guidance_scale: Optional[float] = None,
        num_sampling_steps: Optional[int] = None,
    ) -> Tuple[List[str], Dict[str, int]]:
        """Generate molecules with hard chemical validity checks.

        Repeats generation until *n_molecules* valid SMILES are collected,
        or until a maximum number of total attempts is exceeded.

        Returns:
            (valid_smiles_list, rejection_stats_dict)
        """
        from data.molecular_graph import graph_to_smiles
        from rdkit import Chem
        from rdkit.Chem import Descriptors

        valid_molecules: List[str] = []
        stats = {
            'attempts': 0,
            'invalid_graph': 0,
            'sanitization_fail': 0,
            'property_reject': 0,
            'success': 0,
        }

        max_total_attempts = n_molecules * max_attempts_per_molecule

        while len(valid_molecules) < n_molecules and stats['attempts'] < max_total_attempts:
            # Generate a batch
            remaining = n_molecules - len(valid_molecules)
            batch_size = min(remaining * 2, self.config.generation.batch_size)

            if target_properties is not None:
                graphs = self.guided_sample(
                    num_molecules=batch_size,
                    num_atoms=num_atoms,
                    temperature=temperature,
                    target_properties=target_properties,
                    guidance_scale=guidance_scale,
                    num_sampling_steps=num_sampling_steps,
                )
            else:
                graphs = self.sample(
                    num_molecules=batch_size,
                    num_atoms=num_atoms,
                    temperature=temperature,
                    num_sampling_steps=num_sampling_steps,
                )

            for graph in graphs:
                stats['attempts'] += 1
                if len(valid_molecules) >= n_molecules:
                    break

                smiles = graph_to_smiles(graph)
                if smiles is None:
                    stats['invalid_graph'] += 1
                    continue

                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    stats['invalid_graph'] += 1
                    continue

                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    stats['sanitization_fail'] += 1
                    continue

                if not self.check_property_bounds(mol):
                    stats['property_reject'] += 1
                    continue

                canonical = Chem.MolToSmiles(mol, canonical=True)
                valid_molecules.append(canonical)
                stats['success'] += 1

        # Report
        total = max(stats['attempts'], 1)
        print("\n=== Rejection Sampling Statistics ===")
        print(f"Total attempts:        {stats['attempts']}")
        print(f"Success rate:          {stats['success']/total*100:.1f}%")
        print(f"Invalid graph/SMILES:  {stats['invalid_graph']/total*100:.1f}%")
        print(f"Sanitization failures: {stats['sanitization_fail']/total*100:.1f}%")
        print(f"Property rejections:   {stats['property_reject']/total*100:.1f}%")
        print(f"Valid molecules:       {len(valid_molecules)}/{n_molecules}")

        return valid_molecules, stats

    @staticmethod
    def check_property_bounds(mol) -> bool:
        """Hard bounds for molecular properties. Returns True if acceptable."""
        from rdkit.Chem import Descriptors

        hbd = Descriptors.NumHDonors(mol)
        hba = Descriptors.NumHAcceptors(mol)
        tpsa = Descriptors.TPSA(mol)
        aromatic_rings = Descriptors.NumAromaticRings(mol)
        heavy_atoms = mol.GetNumHeavyAtoms()
        mw = Descriptors.MolWt(mol)
        logp = Descriptors.MolLogP(mol)

        # Reject chemically impossible / non-drug-like
        if hbd == 0 and hba > 15:
            return False
        if hbd > 10:
            return False
        if hba > 15:
            return False
        if tpsa > 200:
            return False
        if aromatic_rings == 0:
            return False
        if heavy_atoms < 10 or heavy_atoms > 40:
            return False
        if not (200 <= mw <= 550):
            return False
        if not (-1 <= logp <= 6):
            return False

        return True

### 7.2 · Post-Processing
Convert generated graphs to SMILES, filter, deduplicate, and save.

In [ ]:
"""
Post-processing for generated molecules.
"""
from typing import List, Optional, Set
import os
import csv
from rdkit import Chem

import sys

# Add project root to path for imports

# (molecular_graph functions defined above)
# (chemistry utils defined above)
# (filter functions defined above)

def postprocess_molecule(graph) -> Optional[str]:
    """
    Convert generated graph to valid SMILES.

    Args:
        graph: PyG Data object

    Returns:
        Canonical SMILES string or None
    """
    smiles = graph_to_smiles(graph)
    if smiles is None:
        return None

    canonical = canonicalize_smiles(smiles)
    if canonical is None:
        return None

    if not is_valid_molecule(canonical):
        return None

    return canonical

def filter_generated(
    smiles_list: List[str],
    drug_like: bool = True,
    glue_like: bool = True,
    pains: bool = True,
    sa_accessible: bool = True,
    no_reactive: bool = True,
    max_heavy_atoms: int = 35,
    min_aromatic_rings: int = 1,
    min_hbd_hba: int = 2,
    min_fsp3: float = 0.2,
) -> List[str]:
    """
    Filter generated molecules by multiple criteria.

    Args:
        smiles_list: List of SMILES strings
        drug_like: Apply Lipinski's Rule of Five
        glue_like: Apply glue-likeness criteria
        pains: Apply PAINS filter
        sa_accessible: Check synthetic accessibility
        no_reactive: Reject molecules with reactive groups
        max_heavy_atoms: Maximum heavy atom count
        min_aromatic_rings: Minimum aromatic ring count
        min_hbd_hba: Minimum HBD + HBA
        min_fsp3: Minimum fraction sp3 carbons

    Returns:
        Filtered list of SMILES
    """
    filtered = []

    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        # Heavy atom check
        if mol.GetNumHeavyAtoms() > max_heavy_atoms:
            continue

        # Aromatic ring check
        from rdkit.Chem import Descriptors
        if Descriptors.NumAromaticRings(mol) < min_aromatic_rings:
            continue

        # HBD + HBA check
        from rdkit.Chem import rdMolDescriptors
        hbd = rdMolDescriptors.CalcNumHBD(mol)
        hba = rdMolDescriptors.CalcNumHBA(mol)
        if (hbd + hba) < min_hbd_hba:
            continue

        # Fsp3 check
        props = get_molecular_properties(smiles)
        if props and props.get('fraction_sp3', 0) < min_fsp3:
            continue

        # Standard filters
        if drug_like and not is_drug_like(mol):
            continue
        if glue_like and not is_glue_like(mol):
            continue
        if pains and not passes_pains_filter(mol):
            continue
        if sa_accessible and not is_synthetically_accessible(mol):
            continue
        if no_reactive and has_reactive_groups(mol):
            continue

        filtered.append(smiles)

    return filtered

def deduplicate(
    smiles_list: List[str],
    training_smiles: Optional[List[str]] = None,
) -> List[str]:
    """
    Remove duplicate molecules and optionally training set molecules.

    Args:
        smiles_list: Generated SMILES
        training_smiles: Training set SMILES to exclude

    Returns:
        Deduplicated list
    """
    seen: Set[str] = set()

    # Add training set
    if training_smiles:
        for s in training_smiles:
            c = canonicalize_smiles(s)
            if c:
                seen.add(c)

    unique = []
    for smiles in smiles_list:
        canonical = canonicalize_smiles(smiles)
        if canonical and canonical not in seen:
            seen.add(canonical)
            unique.append(canonical)

    return unique

def save_molecules(
    smiles_list: List[str],
    output_path: str,
    include_properties: bool = True,
):
    """
    Save molecules to CSV file with properties.

    Args:
        smiles_list: List of SMILES
        output_path: Path to output CSV
        include_properties: Include calculated properties
    """
    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else '.', exist_ok=True)

    headers = ['smiles']
    if include_properties:
        headers.extend([
            'molecular_weight', 'logp', 'hbd', 'hba', 'tpsa',
            'rotatable_bonds', 'num_rings', 'num_aromatic_rings',
            'num_heavy_atoms', 'fraction_sp3', 'qed',
        ])

    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(headers)

        for smiles in smiles_list:
            row = [smiles]
            if include_properties:
                props = get_molecular_properties(smiles)
                if props:
                    row.extend([
                        f"{props.get('molecular_weight', 0):.2f}",
                        f"{props.get('logp', 0):.2f}",
                        props.get('hbd', 0),
                        props.get('hba', 0),
                        f"{props.get('tpsa', 0):.2f}",
                        props.get('rotatable_bonds', 0),
                        props.get('num_rings', 0),
                        props.get('num_aromatic_rings', 0),
                        props.get('num_heavy_atoms', 0),
                        f"{props.get('fraction_sp3', 0):.3f}",
                        f"{props.get('qed', 0):.3f}",
                    ])
                else:
                    row.extend([''] * 11)
            writer.writerow(row)

    print(f"Saved {len(smiles_list)} molecules to {output_path}")

def save_molecules_sdf(
    smiles_list: List[str],
    output_path: str,
):
    """
    Save molecules to SDF format.

    Args:
        smiles_list: List of SMILES
        output_path: Path to output SDF file
    """
    from rdkit.Chem import AllChem

    os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else '.', exist_ok=True)

    writer = Chem.SDWriter(output_path)

    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        # Add 2D coordinates
        AllChem.Compute2DCoords(mol)

        # Add properties
        props = get_molecular_properties(smiles)
        if props:
            for key, value in props.items():
                mol.SetProp(key, str(value))

        writer.write(mol)

    writer.close()
    print(f"Saved {len(smiles_list)} molecules to {output_path}")

### 7.3 · Diverse Sampler
Extended sampler with variable sizes, temperatures, guidance scales, and scaffold-diversity tracking.

In [ ]:
"""
Diverse molecule sampling strategies for the molecular diffusion model.

Extends the base ``MoleculeSampler`` with:
    - Variable molecule sizes (15-35 atoms)
    - Variable temperatures (0.7-1.2)
    - Variable guidance scales (1.5-3.0)
    - Multiple diverse property targets
    - Scaffold-tracking to avoid over-representation
"""
from typing import Optional, List, Dict, Tuple
import random

import torch

# (config already defined above)
# (MolecularDiffusion defined above)
# (MoleculeSampler defined above)
# (postprocess functions defined above)
# (chemistry utils defined above)

class DiverseMoleculeSampler(MoleculeSampler):
    """
    Sampler with diversity-promoting strategies layered on top of the
    base ``MoleculeSampler``.

    Key additions over the base class:

    * ``sample_diverse`` – sweep atom counts, temperatures, and guidance
      scales to cover a wider region of chemical space.
    * ``generate_diverse_property_targets`` – produces multiple target
      property vectors spanning glue-like space.
    * ``scaffold_diversity_sampling`` – caps the number of molecules per
      Murcko scaffold.

    Example::

        sampler = DiverseMoleculeSampler(model, config)
        smiles = sampler.sample_diverse(
            num_molecules=500,
            size_range=(15, 35),
            temp_range=(0.7, 1.2),
            guidance_range=(1.5, 3.0),
        )
    """

    def __init__(self, model: MolecularDiffusion, config: Config, seed: int = 42):
        super().__init__(model, config)
        self.rng = random.Random(seed)

    # ── Public API ──────────────────────────────────────────────────

    def sample_diverse(
        self,
        num_molecules: int = 500,
        size_range: Tuple[int, int] = (15, 35),
        temp_range: Tuple[float, float] = (0.7, 1.2),
        guidance_range: Tuple[float, float] = (1.5, 3.0),
        batch_size: int = 50,
        num_sampling_steps: Optional[int] = None,
    ) -> List[str]:
        """
        Generate molecules by randomly varying atom count, temperature,
        and guidance scale across batches.

        Args:
            num_molecules: Total SMILES wanted.
            size_range: (min_atoms, max_atoms) uniform range.
            temp_range: (min_temp, max_temp) uniform range.
            guidance_range: (min_guide, max_guide) uniform range.
            batch_size: Molecules per inner sampling call.
            num_sampling_steps: Override diffusion steps.

        Returns:
            List of unique, valid canonical SMILES.
        """
        collected: List[str] = []
        seen: set = set()
        total_attempts = 0
        max_attempts = num_molecules * 5  # safety cap

        targets = self.generate_diverse_property_targets(
            max(5, num_molecules // batch_size)
        )

        while len(collected) < num_molecules and total_attempts < max_attempts:
            n_atoms = self.rng.randint(size_range[0], size_range[1])
            temp = round(
                self.rng.uniform(temp_range[0], temp_range[1]), 2
            )
            guide = round(
                self.rng.uniform(guidance_range[0], guidance_range[1]), 2
            )
            target = self.rng.choice(targets)

            remaining = num_molecules - len(collected)
            n_batch = min(batch_size, remaining)

            graphs = self.guided_sample(
                num_molecules=n_batch,
                num_atoms=n_atoms,
                temperature=temp,
                target_properties=target,
                guidance_scale=guide,
                num_sampling_steps=num_sampling_steps,
            )

            for g in graphs:
                smi = postprocess_molecule(g)
                if smi is None:
                    continue
                canon = canonicalize_smiles(smi)
                if canon and canon not in seen:
                    seen.add(canon)
                    collected.append(canon)

            total_attempts += n_batch

        print(
            f"DiverseSampler: collected {len(collected)} unique molecules "
            f"from {total_attempts} attempts "
            f"({len(collected) / max(1, total_attempts) * 100:.1f}% yield)"
        )
        return collected

    def generate_diverse_property_targets(
        self,
        num_targets: int = 10,
    ) -> List[Dict[str, float]]:
        """
        Produce diverse property-target vectors spanning glue-like space.

        The ranges sampled are:
            MW:  250 – 450
            LogP: 1.0 – 4.0
            Aromatic rings: 1 – 3
            HBD: 1 – 4
            HBA: 2 – 8
            Fsp3: 0.15 – 0.55
            TPSA: 40 – 120

        Targets are normalised to the same scale the model was trained on
        (using the ``Config`` normalisation ranges, falling back to
        reasonable defaults).

        Args:
            num_targets: How many target vectors to generate.

        Returns:
            List of property dictionaries ready for ``guided_sample``.
        """
        targets: List[Dict[str, float]] = []

        ranges = {
            "molecular_weight": (250.0, 450.0),
            "logp": (1.0, 4.0),
            "num_aromatic_rings": (1.0, 3.0),
            "hbd": (1.0, 4.0),
            "hba": (2.0, 8.0),
            "fraction_sp3": (0.15, 0.55),
            "tpsa": (40.0, 120.0),
        }

        for _ in range(num_targets):
            t = {}
            for key, (lo, hi) in ranges.items():
                t[key] = round(self.rng.uniform(lo, hi), 2)
            targets.append(t)

        return targets

    def scaffold_diversity_sampling(
        self,
        num_molecules: int = 500,
        max_per_scaffold: int = 5,
        size_range: Tuple[int, int] = (15, 35),
        temperature: float = 1.0,
        guidance_scale: float = 2.0,
        batch_size: int = 50,
        num_sampling_steps: Optional[int] = None,
    ) -> List[str]:
        """
        Generate molecules while capping the number of representatives
        per Murcko scaffold, promoting structural diversity.

        Args:
            num_molecules: Target number of molecules.
            max_per_scaffold: Maximum accept per scaffold.
            size_range: (min, max) atom range.
            temperature: Sampling temperature.
            guidance_scale: Guidance scale.
            batch_size: Molecules per sampling call.
            num_sampling_steps: Override diffusion steps.

        Returns:
            List of unique SMILES with high scaffold diversity.
        """
        from collections import Counter

        collected: List[str] = []
        seen: set = set()
        scaffold_counts: Counter = Counter()
        total_attempts = 0
        max_attempts = num_molecules * 8

        targets = self.generate_diverse_property_targets(
            max(5, num_molecules // batch_size)
        )

        while len(collected) < num_molecules and total_attempts < max_attempts:
            n_atoms = self.rng.randint(size_range[0], size_range[1])
            target = self.rng.choice(targets)

            remaining = num_molecules - len(collected)
            n_batch = min(batch_size, remaining * 2)  # oversample

            graphs = self.guided_sample(
                num_molecules=n_batch,
                num_atoms=n_atoms,
                temperature=temperature,
                target_properties=target,
                guidance_scale=guidance_scale,
                num_sampling_steps=num_sampling_steps,
            )

            for g in graphs:
                if len(collected) >= num_molecules:
                    break

                smi = postprocess_molecule(g)
                if smi is None:
                    continue
                canon = canonicalize_smiles(smi)
                if not canon or canon in seen:
                    continue

                scaffold = get_murcko_scaffold(canon) or "none"
                if scaffold_counts[scaffold] >= max_per_scaffold:
                    continue  # skip over-represented scaffold

                seen.add(canon)
                scaffold_counts[scaffold] += 1
                collected.append(canon)

            total_attempts += n_batch

        n_scaffolds = len(scaffold_counts)
        diversity = n_scaffolds / max(1, len(collected))
        print(
            f"ScaffoldDiversity: {len(collected)} molecules, "
            f"{n_scaffolds} scaffolds ({diversity:.1%} diversity), "
            f"{total_attempts} attempts"
        )
        return collected

## 8 · Evaluation
### 8.1 · Metrics
Validity, uniqueness, novelty, diversity, scaffold diversity, QED, Wasserstein distance, and glue similarity.

In [ ]:
"""
Evaluation metrics for molecular generation.
"""
from typing import List, Optional, Dict
from collections import Counter
import numpy as np

# (chemistry utils defined above)
from rdkit.Chem import DataStructs

def validity_rate(molecules: List[str]) -> float:
    """Fraction of molecules that are chemically valid."""
    if not molecules:
        return 0.0
    valid = sum(1 for m in molecules if is_valid_molecule(m))
    return valid / len(molecules)

def uniqueness_rate(molecules: List[str]) -> float:
    """Fraction of unique molecules among valid ones."""
    valid = [canonicalize_smiles(m) for m in molecules if is_valid_molecule(m)]
    valid = [m for m in valid if m is not None]
    if not valid:
        return 0.0
    return len(set(valid)) / len(valid)

def novelty_rate(molecules: List[str], training_set: List[str]) -> float:
    """Fraction of generated molecules not in training set."""
    train_canonical = set()
    for m in training_set:
        c = canonicalize_smiles(m)
        if c:
            train_canonical.add(c)

    valid = [canonicalize_smiles(m) for m in molecules if is_valid_molecule(m)]
    valid = [m for m in valid if m is not None]

    if not valid:
        return 0.0

    novel = sum(1 for m in valid if m not in train_canonical)
    return novel / len(valid)

def internal_diversity(molecules: List[str], sample_size: int = 1000) -> float:
    """
    Average pairwise Tanimoto distance among generated molecules.
    1.0 = maximally diverse, 0.0 = all identical.
    """
    valid = [m for m in molecules if is_valid_molecule(m)]
    if len(valid) < 2:
        return 0.0

    fps = [get_morgan_fingerprint(m) for m in valid]
    fps = [fp for fp in fps if fp is not None]

    if len(fps) < 2:
        return 0.0

    # Sample pairs if too many
    if len(fps) > sample_size:
        indices = np.random.choice(len(fps), sample_size, replace=False)
        fps = [fps[i] for i in indices]

    similarities = []
    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
            similarities.append(sim)

    avg_sim = np.mean(similarities)
    return 1.0 - avg_sim

def scaffold_diversity(molecules: List[str]) -> float:
    """
    Fraction of unique Murcko scaffolds among valid molecules.

    Higher = more scaffold-diverse output.
    """
    valid = [m for m in molecules if is_valid_molecule(m)]
    if not valid:
        return 0.0

    scaffolds = []
    for m in valid:
        s = get_murcko_scaffold(m)
        if s is not None:
            scaffolds.append(s)

    if not scaffolds:
        return 0.0

    return len(set(scaffolds)) / len(scaffolds)

def wasserstein_property_distance(
    generated: List[str],
    reference: List[str],
    property_name: str,
) -> float:
    """
    1D Wasserstein (earth mover's) distance between property distributions.

    Args:
        generated: Generated SMILES
        reference: Reference SMILES
        property_name: Property to compare (key in get_molecular_properties)

    Returns:
        Wasserstein distance (lower = more similar distributions)
    """
    from scipy.stats import wasserstein_distance

    gen_values = []
    for m in generated:
        props = get_molecular_properties(m)
        if props and property_name in props:
            gen_values.append(props[property_name])

    ref_values = []
    for m in reference:
        props = get_molecular_properties(m)
        if props and property_name in props:
            ref_values.append(props[property_name])

    if not gen_values or not ref_values:
        return float('inf')

    return wasserstein_distance(gen_values, ref_values)

def compute_glue_similarity(
    generated: List[str],
    known_glues: List[str],
) -> Dict[str, float]:
    """
    Compute similarity metrics between generated and known glue molecules.

    Returns:
        Dict with mean_nn_sim (mean nearest-neighbor Tanimoto),
        max_sim, fraction with sim > 0.4
    """
    gen_fps = [get_morgan_fingerprint(m) for m in generated if is_valid_molecule(m)]
    gen_fps = [fp for fp in gen_fps if fp is not None]

    glue_fps = [get_morgan_fingerprint(m) for m in known_glues if is_valid_molecule(m)]
    glue_fps = [fp for fp in glue_fps if fp is not None]

    if not gen_fps or not glue_fps:
        return {'mean_nn_sim': 0.0, 'max_sim': 0.0, 'frac_similar': 0.0}

    nn_sims = []
    for gen_fp in gen_fps:
        max_sim = max(DataStructs.TanimotoSimilarity(gen_fp, g) for g in glue_fps)
        nn_sims.append(max_sim)

    return {
        'mean_nn_sim': float(np.mean(nn_sims)),
        'max_sim': float(np.max(nn_sims)),
        'frac_similar': float(np.mean([s > 0.4 for s in nn_sims])),
    }

def qed_score_stats(molecules: List[str]) -> Dict[str, float]:
    """
    QED score statistics for generated molecules.
    """
    scores = [calculate_qed(m) for m in molecules if is_valid_molecule(m)]
    scores = [s for s in scores if s > 0]

    if not scores:
        return {'qed_mean': 0.0, 'qed_std': 0.0, 'qed_frac_good': 0.0}

    return {
        'qed_mean': float(np.mean(scores)),
        'qed_std': float(np.std(scores)),
        'qed_frac_good': float(np.mean([s > 0.4 for s in scores])),
    }

def property_distribution(
    molecules: List[str],
    property_name: str,
) -> Dict[str, float]:
    """Get statistics for a property across generated molecules."""
    values = []
    for m in molecules:
        props = get_molecular_properties(m)
        if props and property_name in props:
            values.append(props[property_name])

    if not values:
        return {'mean': 0.0, 'std': 0.0, 'min': 0.0, 'max': 0.0, 'n': 0}

    return {
        'mean': float(np.mean(values)),
        'std': float(np.std(values)),
        'min': float(np.min(values)),
        'max': float(np.max(values)),
        'n': len(values),
    }

def compute_all_metrics(
    generated: List[str],
    training_set: Optional[List[str]] = None,
    known_glues: Optional[List[str]] = None,
) -> Dict[str, any]:
    """
    Compute all evaluation metrics.

    Args:
        generated: Generated SMILES
        training_set: Training SMILES for novelty
        known_glues: Known glue SMILES for similarity metrics

    Returns:
        Dictionary of all metrics
    """
    metrics = {
        'validity': validity_rate(generated),
        'uniqueness': uniqueness_rate(generated),
        'diversity': internal_diversity(generated),
        'scaffold_diversity': scaffold_diversity(generated),
    }

    if training_set:
        metrics['novelty'] = novelty_rate(generated, training_set)

    # QED stats
    metrics.update(qed_score_stats(generated))

    # Property distributions
    for prop in ['molecular_weight', 'logp', 'num_aromatic_rings', 'hbd', 'hba', 'tpsa', 'fraction_sp3', 'qed']:
        dist = property_distribution(generated, prop)
        metrics[f'{prop}_mean'] = dist['mean']
        metrics[f'{prop}_std'] = dist['std']

    # Glue similarity
    if known_glues:
        metrics.update(compute_glue_similarity(generated, known_glues))

    # Wasserstein distances to training set
    if training_set:
        for prop in ['molecular_weight', 'logp', 'tpsa']:
            try:
                metrics[f'{prop}_wasserstein'] = wasserstein_property_distance(
                    generated, training_set, prop
                )
            except ImportError:
                pass  # scipy not available

    return metrics

def print_metrics_report(metrics: Dict[str, any]):
    """Print a formatted metrics report."""
    print("\n" + "=" * 60)
    print("GENERATION METRICS REPORT")
    print("=" * 60)

    # Core metrics
    print("\n--- Core Metrics ---")
    for key in ['validity', 'uniqueness', 'novelty', 'diversity', 'scaffold_diversity']:
        if key in metrics:
            print(f"  {key:25s}: {metrics[key]:.4f}")

    # QED
    print("\n--- QED ---")
    for key in ['qed_mean', 'qed_std', 'qed_frac_good']:
        if key in metrics:
            print(f"  {key:25s}: {metrics[key]:.4f}")

    # Glue similarity
    if 'mean_nn_sim' in metrics:
        print("\n--- Glue Similarity ---")
        for key in ['mean_nn_sim', 'max_sim', 'frac_similar']:
            print(f"  {key:25s}: {metrics[key]:.4f}")

    # Property distributions
    print("\n--- Property Distributions ---")
    for prop in ['molecular_weight', 'logp', 'num_aromatic_rings', 'hbd', 'hba', 'tpsa', 'fraction_sp3', 'qed']:
        mean_key = f'{prop}_mean'
        std_key = f'{prop}_std'
        if mean_key in metrics:
            print(f"  {prop:25s}: {metrics[mean_key]:.2f} ± {metrics.get(std_key, 0):.2f}")

    # Wasserstein
    wass_keys = [k for k in metrics if k.endswith('_wasserstein')]
    if wass_keys:
        print("\n--- Wasserstein Distances ---")
        for key in wass_keys:
            print(f"  {key:25s}: {metrics[key]:.4f}")

    print("\n" + "=" * 60)

### 8.2 · Visualization
Property distributions, molecule grids, scatter matrices, scaffold charts, correlation heatmaps, and full reports.

In [ ]:
"""
Comprehensive visualization for comparing generated and training molecules.

Functions:
    plot_property_distributions  – overlay histograms + KDE
    plot_molecule_grid           – 2D molecule grid coloured by QED
    plot_scatter_matrix          – pairwise scatter by molecule type
    plot_scaffold_distribution   – top-N Murcko scaffold bar chart
    plot_correlation_heatmap     – property correlation heatmap
    generate_full_report         – run everything and save to output_dir
"""
import os
import sys
from typing import List, Optional

import numpy as np
import pandas as pd

# (chemistry utils defined above)

# ── Property calculation helpers ────────────────────────────────────

def _smiles_to_props_df(smiles_list: List[str], label: str = "") -> pd.DataFrame:
    """Convert a list of SMILES to a properties DataFrame."""
    records = []
    for smi in smiles_list:
        props = get_molecular_properties(smi)
        if props is not None:
            props["smiles"] = smi
            props["scaffold"] = get_murcko_scaffold(smi) or "none"
            if label:
                props["source"] = label
            records.append(props)
    return pd.DataFrame(records)

# ── 1. Property Distributions ──────────────────────────────────────

PROPERTY_DISPLAY = {
    "molecular_weight": "Molecular Weight (Da)",
    "logp": "LogP",
    "hbd": "H-Bond Donors",
    "hba": "H-Bond Acceptors",
    "tpsa": "TPSA (Å²)",
    "num_rings": "Total Rings",
    "num_aromatic_rings": "Aromatic Rings",
    "fraction_sp3": "Fraction sp³",
    "qed": "QED",
}

def plot_property_distributions(
    gen_smiles: List[str],
    train_smiles: List[str],
    output_file: str = "property_distributions.png",
):
    """
    Overlay histograms and KDE curves of key properties for generated
    vs. training molecules.

    Args:
        gen_smiles: Generated SMILES list.
        train_smiles: Training SMILES list.
        output_file: Path for saved figure.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    df_gen = _smiles_to_props_df(gen_smiles, "Generated")
    df_train = _smiles_to_props_df(train_smiles, "Training")

    props = list(PROPERTY_DISPLAY.keys())

    n_props = len(props)
    ncols = 3
    nrows = (n_props + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()

    for idx, prop in enumerate(props):
        ax = axes[idx]
        if prop in df_gen.columns:
            sns.histplot(
                df_gen[prop].dropna(), ax=ax, label="Generated",
                kde=True, stat="density", alpha=0.45, color="#4C72B0",
            )
            g_mean = df_gen[prop].mean()
            g_std = df_gen[prop].std()
            ax.axvline(g_mean, color="#4C72B0", ls="--", lw=1)
            ax.text(
                0.98, 0.95,
                f"Gen: {g_mean:.2f}±{g_std:.2f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7, color="#4C72B0",
            )
        if prop in df_train.columns:
            sns.histplot(
                df_train[prop].dropna(), ax=ax, label="Training",
                kde=True, stat="density", alpha=0.40, color="#DD8452",
            )
            t_mean = df_train[prop].mean()
            t_std = df_train[prop].std()
            ax.axvline(t_mean, color="#DD8452", ls="--", lw=1)
            ax.text(
                0.98, 0.85,
                f"Train: {t_mean:.2f}±{t_std:.2f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7, color="#DD8452",
            )
        ax.set_title(PROPERTY_DISPLAY.get(prop, prop))
        ax.legend(fontsize=7)

    # Hide unused axes
    for j in range(idx + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Property Distributions: Generated vs Training", fontsize=14, y=1.01)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    fig.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved property distributions to {output_file}")

# ── 2. Molecule Grid ───────────────────────────────────────────────

def plot_molecule_grid(
    smiles_list: List[str],
    output_file: str = "molecule_grid.png",
    n_mols: int = 20,
    mols_per_row: int = 5,
):
    """
    Render a grid of 2D molecule structures colour-coded by QED score.

    Args:
        smiles_list: SMILES to draw.
        output_file: Output image path.
        n_mols: Maximum number of molecules to show.
        mols_per_row: Molecules per row in the grid.
    """
    from rdkit import Chem
    from rdkit.Chem import Draw, AllChem

    mols, legends = [], []
    for smi in smiles_list[:n_mols]:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        AllChem.Compute2DCoords(mol)
        qed = calculate_qed(mol)
        mols.append(mol)
        legends.append(f"QED={qed:.2f}")

    if not mols:
        print("No valid molecules to draw.")
        return

    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=mols_per_row,
        subImgSize=(300, 300),
        legends=legends,
    )

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    img.save(output_file)
    print(f"Saved molecule grid ({len(mols)} mols) to {output_file}")

# ── 3. Scatter Matrix ──────────────────────────────────────────────

def plot_scatter_matrix(
    gen_smiles: List[str],
    train_smiles: List[str],
    output_file: str = "scatter_matrix.png",
    glue_smiles: Optional[List[str]] = None,
):
    """
    Pairwise scatter plots of key properties coloured by molecule type.

    Args:
        gen_smiles: Generated SMILES.
        train_smiles: Training SMILES.
        output_file: Output image path.
        glue_smiles: Optional known-glue SMILES (plotted separately).
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    df_gen = _smiles_to_props_df(gen_smiles, "Generated")
    df_train = _smiles_to_props_df(train_smiles, "Training")
    dfs = [df_gen, df_train]
    if glue_smiles:
        df_glue = _smiles_to_props_df(glue_smiles, "Known Glues")
        dfs.append(df_glue)

    df_all = pd.concat(dfs, ignore_index=True)

    props = ["molecular_weight", "logp", "tpsa", "num_aromatic_rings", "qed"]
    available = [p for p in props if p in df_all.columns]

    if len(available) < 2:
        print("Not enough properties to create scatter matrix.")
        return

    palette = {"Generated": "#4C72B0", "Training": "#DD8452", "Known Glues": "#55A868"}

    g = sns.pairplot(
        df_all,
        vars=available,
        hue="source",
        palette=palette,
        diag_kind="kde",
        plot_kws={"alpha": 0.4, "s": 12},
        height=2.2,
    )
    g.fig.suptitle("Property Scatter Matrix", y=1.02)
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    g.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.close(g.fig)
    print(f"Saved scatter matrix to {output_file}")

# ── 4. Scaffold Distribution ───────────────────────────────────────

def plot_scaffold_distribution(
    gen_smiles: List[str],
    output_file: str = "scaffold_distribution.png",
    train_smiles: Optional[List[str]] = None,
    top_n: int = 15,
):
    """
    Bar chart of top-N Murcko scaffolds.

    Args:
        gen_smiles: Generated SMILES.
        output_file: Output image path.
        train_smiles: Optional training SMILES for comparison.
        top_n: Number of top scaffolds to show.
    """
    import matplotlib.pyplot as plt
    from collections import Counter

    gen_scaffolds = [get_murcko_scaffold(s) or "none" for s in gen_smiles]
    gen_counts = Counter(gen_scaffolds).most_common(top_n)

    scaffolds = [s for s, _ in gen_counts]
    gen_vals = [c for _, c in gen_counts]

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.35)))
    x = np.arange(len(scaffolds))
    width = 0.35

    ax.barh(x, gen_vals, width, label="Generated", color="#4C72B0", alpha=0.8)

    if train_smiles:
        train_scaffolds = [get_murcko_scaffold(s) or "none" for s in train_smiles]
        train_counter = Counter(train_scaffolds)
        train_vals = [train_counter.get(s, 0) for s in scaffolds]
        ax.barh(x + width, train_vals, width, label="Training", color="#DD8452", alpha=0.8)

    ax.set_yticks(x + width / 2)
    # Truncate long SMILES for labels
    labels = [s[:30] + "…" if len(s) > 30 else s for s in scaffolds]
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel("Count")
    ax.set_title(f"Top {top_n} Murcko Scaffolds")
    ax.legend()
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    fig.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved scaffold distribution to {output_file}")

# ── 5. Correlation Heatmap ─────────────────────────────────────────

def plot_correlation_heatmap(
    smiles_list: List[str],
    output_file: str = "correlation_heatmap.png",
    label: str = "",
):
    """
    Heatmap of pairwise correlations between molecular properties.

    Args:
        smiles_list: SMILES list.
        output_file: Output image path.
        label: Title annotation (e.g. "Generated" or "Training").
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    df = _smiles_to_props_df(smiles_list)
    props = [
        "molecular_weight", "logp", "hbd", "hba", "tpsa",
        "num_aromatic_rings", "rotatable_bonds", "fraction_sp3", "qed",
    ]
    available = [p for p in props if p in df.columns]
    if len(available) < 3:
        print("Not enough properties for correlation heatmap.")
        return

    corr = df[available].corr()

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(
        corr, annot=True, fmt=".2f", cmap="RdBu_r",
        center=0, vmin=-1, vmax=1, ax=ax, square=True,
        linewidths=0.5,
    )
    title = "Property Correlations"
    if label:
        title += f" ({label})"
    ax.set_title(title)
    plt.tight_layout()
    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    fig.savefig(output_file, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved correlation heatmap to {output_file}")

# ── 6. Full Report ─────────────────────────────────────────────────

def generate_full_report(
    gen_smiles: List[str],
    train_smiles: List[str],
    output_dir: str = "outputs/report",
    glue_smiles: Optional[List[str]] = None,
):
    """
    Generate all visualisation outputs into *output_dir*.

    Args:
        gen_smiles: Generated SMILES.
        train_smiles: Training SMILES.
        output_dir: Directory for all output images.
        glue_smiles: Optional known-glue SMILES for extra comparison.
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'='*50}")
    print(f"GENERATING FULL REPORT → {output_dir}/")
    print(f"{'='*50}\n")

    plot_property_distributions(
        gen_smiles, train_smiles,
        os.path.join(output_dir, "property_distributions.png"),
    )

    plot_molecule_grid(
        gen_smiles,
        os.path.join(output_dir, "molecule_grid_generated.png"),
    )

    plot_scatter_matrix(
        gen_smiles, train_smiles,
        os.path.join(output_dir, "scatter_matrix.png"),
        glue_smiles=glue_smiles,
    )

    plot_scaffold_distribution(
        gen_smiles,
        os.path.join(output_dir, "scaffold_distribution.png"),
        train_smiles=train_smiles,
    )

    plot_correlation_heatmap(
        gen_smiles,
        os.path.join(output_dir, "correlation_heatmap_generated.png"),
        label="Generated",
    )
    plot_correlation_heatmap(
        train_smiles,
        os.path.join(output_dir, "correlation_heatmap_training.png"),
        label="Training",
    )

    print(f"\n✅ Full report saved to {output_dir}/")

### 8.3 · Analysis
High-level analysis utilities for computing properties and generating summary reports.

In [ ]:
"""
Analysis and visualization for generated molecules.
"""
import os
from typing import Optional, List
import pandas as pd
import numpy as np

import sys

# (chemistry utils defined above)


def analyze_molecules(smiles_list: List[str]) -> pd.DataFrame:
    """Compute properties for a list of molecules."""
    records = []
    for smiles in smiles_list:
        props = get_molecular_properties(smiles)
        if props:
            props['smiles'] = smiles
            props['scaffold'] = get_murcko_scaffold(smiles) or 'unknown'
            records.append(props)
    return pd.DataFrame(records)

def plot_property_distributions(
    df_gen: pd.DataFrame,
    df_train: Optional[pd.DataFrame] = None,
    output_dir: str = 'analysis',
):
    """Plot property distributions."""
    import matplotlib.pyplot as plt
    import seaborn as sns

    os.makedirs(output_dir, exist_ok=True)

    properties = [
        'molecular_weight', 'logp', 'tpsa', 'hbd', 'hba',
        'num_aromatic_rings', 'rotatable_bonds', 'fraction_sp3', 'qed',
    ]

    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()

    for idx, prop in enumerate(properties):
        if prop not in df_gen.columns:
            continue

        ax = axes[idx]
        sns.histplot(df_gen[prop], ax=ax, label='Generated', kde=True, alpha=0.5, color='blue')
        if df_train is not None and prop in df_train.columns:
            sns.histplot(df_train[prop], ax=ax, label='Training', kde=True, alpha=0.5, color='orange')
        ax.set_title(prop.replace('_', ' ').title())
        ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'property_distributions.png'), dpi=150)
    plt.close()
    print(f"Saved property distributions to {output_dir}/property_distributions.png")

def plot_scatter_matrix(df: pd.DataFrame, output_dir: str = 'analysis'):
    """Plot scatter matrix of key properties."""
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)

    props = ['molecular_weight', 'logp', 'tpsa', 'num_aromatic_rings', 'qed']
    available = [p for p in props if p in df.columns]

    if len(available) >= 2:
        pd.plotting.scatter_matrix(df[available], figsize=(12, 12), alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'scatter_matrix.png'), dpi=150)
        plt.close()
        print(f"Saved scatter matrix to {output_dir}/scatter_matrix.png")

def plot_scaffold_distribution(df: pd.DataFrame, output_dir: str = 'analysis', top_n: int = 15):
    """Plot distribution of top Murcko scaffolds."""
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)

    if 'scaffold' not in df.columns:
        return

    scaffold_counts = df['scaffold'].value_counts().head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    scaffold_counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Count')
    ax.set_title(f'Top {top_n} Murcko Scaffolds')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'scaffold_distribution.png'), dpi=150)
    plt.close()
    print(f"Saved scaffold distribution to {output_dir}/scaffold_distribution.png")

def print_summary_statistics(df: pd.DataFrame, label: str = "Generated"):
    """Print summary statistics."""
    print(f"\n--- {label} Summary ({len(df)} molecules) ---")

    props = [
        'molecular_weight', 'logp', 'tpsa', 'hbd', 'hba',
        'num_aromatic_rings', 'rotatable_bonds', 'fraction_sp3', 'qed',
    ]

    for prop in props:
        if prop in df.columns:
            values = df[prop].dropna()
            print(f"  {prop:25s}: {values.mean():.2f} ± {values.std():.2f} "
                  f"[{values.min():.2f}, {values.max():.2f}]")

    if 'scaffold' in df.columns:
        n_unique = df['scaffold'].nunique()
        print(f"  {'unique_scaffolds':25s}: {n_unique} ({n_unique / len(df) * 100:.1f}%)")

---
## 9 · Run the Pipeline
The cells below wire everything together for a complete train → generate → evaluate workflow.
Adjust the configuration variables at the top of each cell to suit your hardware and data.

### 9.1 · Prepare Training Data

In [ ]:
# ── Download 50k drug-like molecules from ChEMBL ─────────────────
# This cell downloads ~50,000 drug-like molecules from ChEMBL to use
# as training data alongside the glue chemotypes.
# ⚡ Takes ~5-15 minutes depending on network speed.
# If ChEMBL is unavailable, falls back to existing glue_chemotypes.csv.

!pip install -q chembl_webresource_client

import os
import pandas as pd

CHEMBL_CSV = os.path.join(DATA_DIR, "chembl_druglike.csv")
GLUE_CSV   = os.path.join(DATA_DIR, "glue_chemotypes.csv")
TRAINING_DATA_PATH = os.path.join(DATA_DIR, "training_data.csv")

# --- Download from ChEMBL ---
if os.path.exists(CHEMBL_CSV):
    print(f"✅ ChEMBL data already exists at {CHEMBL_CSV}, skipping download.")
    df_chembl = pd.read_csv(CHEMBL_CSV)
    print(f"   {len(df_chembl)} molecules loaded.")
else:
    print("📥 Downloading ~50,000 drug-like molecules from ChEMBL...")
    chembl_smiles = download_chembl_druglike(n_molecules=50_000)

    if chembl_smiles:
        # Validate with RDKit
        chembl_smiles = validate_smiles_list(chembl_smiles)
        df_chembl = pd.DataFrame({"smiles": chembl_smiles})
        os.makedirs(os.path.dirname(CHEMBL_CSV), exist_ok=True)
        df_chembl.to_csv(CHEMBL_CSV, index=False)
        print(f"✅ Saved {len(df_chembl)} validated molecules to {CHEMBL_CSV}")
    else:
        print("⚠️  ChEMBL download failed. Will use existing data only.")
        df_chembl = pd.DataFrame(columns=["smiles"])

# --- Merge with glue chemotypes ---
if os.path.exists(GLUE_CSV):
    df_glues = pd.read_csv(GLUE_CSV)
    print(f"🧬 Glue chemotypes: {len(df_glues)} molecules")
else:
    df_glues = pd.DataFrame(columns=["smiles"])
    print("ℹ️  No glue_chemotypes.csv found – using ChEMBL data only.")

# Combine & deduplicate
all_smiles = list(set(
    df_chembl["smiles"].dropna().tolist() +
    df_glues["smiles"].dropna().tolist()
))
df_train = pd.DataFrame({"smiles": all_smiles})
os.makedirs(os.path.dirname(TRAINING_DATA_PATH) or ".", exist_ok=True)
df_train.to_csv(TRAINING_DATA_PATH, index=False)

print(f"\n✅ Training data ready: {len(df_train)} unique molecules")
print(f"   Saved to: {TRAINING_DATA_PATH}")

### 9.2 · Configure & Load Dataset

In [ ]:
# ── Build configuration ───────────────────────────────────────────
# Adjust these hyperparameters as needed for your hardware.

config = Config(
    model=ModelConfig(
        hidden_dim=256,
        num_layers=6,
        num_heads=8,
        dropout=0.1,
    ),
    diffusion=DiffusionConfig(
        num_timesteps=500,
        beta_start=1e-4,
        beta_end=0.02,
        beta_schedule="cosine",
    ),
    loss=LossConfig(
        lambda_valency=0.1,
        lambda_property=0.05,
        lambda_fragment=0.01,
    ),
    guidance=GuidanceConfig(
        guidance_scale=2.0,
        condition_dropout=0.1,
    ),
    training=TrainingConfig(
        learning_rate=1e-4,
        batch_size=32,
        epochs=100,
        warmup_steps=1000,
        ema_decay=0.999,
        gradient_clip=1.0,
        save_frequency=10,
        val_frequency=5,
    ),
    generation=GenerationConfig(
        num_molecules=100,
        batch_size=10,
        temperature=0.8,
        min_atoms=15,
        max_atoms=35,
    ),
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=SEED,
)

print("Configuration created:")
print(f"  Model: {config.model.hidden_dim}d, {config.model.num_layers} layers, {config.model.num_heads} heads")
print(f"  Diffusion: {config.diffusion.num_timesteps} timesteps, {config.diffusion.beta_schedule} schedule")
print(f"  Training: lr={config.training.learning_rate}, batch={config.training.batch_size}, epochs={config.training.epochs}")
print(f"  Device: {config.device}")

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
from torch_geometric.loader import DataLoader

# Load molecules from CSV
dataset = MolecularGlueDataset(
    data_path=TRAINING_DATA_PATH,
    filter_drug_like=True,
    filter_glue_like=False,  # Keep broader chemical space for training
    max_atoms=50,
)

print(f"Dataset size: {len(dataset)} molecules")

# Train/validation split
train_dataset, val_dataset = create_train_val_split(dataset, val_fraction=0.1, seed=SEED)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=config.training.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.training.batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# Inspect first batch
batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  Nodes: {batch.x.shape}")
print(f"  Edges: {batch.edge_index.shape}")
print(f"  Edge attrs: {batch.edge_attr.shape}")
print(f"  Num graphs: {batch.num_graphs}")

### 9.3 · Train the Model

In [ ]:
# ── Initialise model & trainer ────────────────────────────────────
model = MolecularDiffusion(config)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

# Trainer requires train_loader (and optionally val_loader) at init time
trainer = Trainer(model, config, train_loader=train_loader, val_loader=val_loader)

# Optional: resume from checkpoint
resume_path = os.path.join(CKPT_DIR, "latest_checkpoint.pt")
if os.path.exists(resume_path):
    trainer.load_checkpoint(resume_path)
    print(f"Resumed from checkpoint")
else:
    print("Starting fresh training")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────
# Adjust epochs for a full run. Use fewer for testing.
NUM_EPOCHS = config.training.epochs  # or set to e.g. 5 for a quick test

history = trainer.train(num_epochs=NUM_EPOCHS)

print("\n✅ Training complete!")

### 9.4 · Generate Molecules

In [ ]:
# ── Generate molecules ────────────────────────────────────────────
model.eval()

sampler = MoleculeSampler(model, config)

# Basic generation
print("Generating molecules...")
graphs = sampler.sample(
    num_molecules=config.generation.num_molecules,
    num_atoms=20,
    temperature=config.generation.temperature,
)

# Post-process: convert graphs → SMILES
raw_smiles = []
for g in graphs:
    smi = postprocess_molecule(g)
    if smi:
        raw_smiles.append(smi)

print(f"\nRaw valid molecules: {len(raw_smiles)}/{len(graphs)}")
print(f"Validity rate: {len(raw_smiles)/max(1,len(graphs))*100:.1f}%")

In [ ]:
# ── Filter & deduplicate ─────────────────────────────────────────
filtered = filter_generated(
    raw_smiles,
    drug_like=True,
    glue_like=True,
    pains=True,
    sa_accessible=True,
    no_reactive=True,
)
print(f"After filtering: {len(filtered)} molecules")

# Load training SMILES for novelty check
train_smiles = dataset.smiles_list if hasattr(dataset, 'smiles_list') else []

unique = deduplicate(filtered, training_smiles=train_smiles if train_smiles else None)
print(f"After deduplication: {len(unique)} novel molecules")

# Save results
output_csv = os.path.join(OUTPUT_DIR, "generated_molecules.csv")
save_molecules(unique, output_csv, include_properties=True)

### 9.5 · Guided Generation (Glue-like Targets)

In [ ]:
# ── Guided generation with glue-like targets ──────────────────────
glue_targets = MoleculeSampler.get_glue_targets()
print("Target properties for glue-like molecules:")
for k, v in glue_targets.items():
    print(f"  {k}: {v:.3f}")

guided_graphs = sampler.guided_sample(
    num_molecules=50,
    num_atoms=22,
    temperature=0.8,
    target_properties=glue_targets,
    guidance_scale=2.0,
)

guided_smiles = [postprocess_molecule(g) for g in guided_graphs]
guided_smiles = [s for s in guided_smiles if s is not None]

guided_filtered = filter_generated(guided_smiles, drug_like=True, glue_like=True)
print(f"\nGuided generation: {len(guided_filtered)} glue-like molecules from {len(guided_graphs)} attempts")

### 9.6 · Evaluate Generated Molecules

In [ ]:
# ── Compute metrics ───────────────────────────────────────────────
all_generated = unique  # Use the deduplicated set

metrics = compute_all_metrics(
    generated=all_generated,
    training_set=train_smiles if train_smiles else None,
    known_glues=None,  # Add known glue SMILES if available
)

print_metrics_report(metrics)

In [ ]:
# ── Glue-likeness scoring ────────────────────────────────────────
if all_generated:
    ranked = rank_molecules_by_glue_score(all_generated[:50])
    print("\nTop 10 molecules by glue-likeness score:")
    print("-" * 60)
    for smi, score in ranked[:10]:
        print(f"  Score: {score:3d}  |  {smi}")

### 9.7 · Visualize Results

In [ ]:
# ── Inline plotting setup ────────────────────────────────────────
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
import matplotlib.pyplot as plt

In [ ]:
# ── Generate full visualisation report ────────────────────────────
if all_generated and train_smiles:
    generate_full_report(
        gen_smiles=all_generated,
        train_smiles=train_smiles,
        output_dir=os.path.join(OUTPUT_DIR, "report"),
    )
else:
    print("Need both generated and training SMILES for full report.")

# ── Show molecule grid inline ─────────────────────────────────────
if all_generated:
    from rdkit import Chem
    from rdkit.Chem import Draw, AllChem
    from IPython.display import display

    mols = []
    legends = []
    for smi in all_generated[:20]:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            AllChem.Compute2DCoords(mol)
            qed = calculate_qed(mol)
            mols.append(mol)
            legends.append(f"QED={qed:.2f}")

    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(300, 300), legends=legends)
        display(img)

---
## 10 · Appendix

### Saving & Loading Checkpoints
```python
# Save
trainer.save_checkpoint("my_model.pt")

# Load
trainer.load_checkpoint("my_model.pt")
```

### Exporting to SDF
```python
save_molecules_sdf(all_generated, os.path.join(OUTPUT_DIR, "molecules.sdf"))
```

### Rejection Sampling
```python
valid_smiles, stats = sampler.generate_with_rejection(
    n_molecules=100,
    max_attempts_per_molecule=10,
    num_atoms=22,
    temperature=0.8,
    target_properties=glue_targets,
    guidance_scale=2.0,
)
```